Can a learned gating function recover part of the 10-point semantic–structured oracle gap by selecting the appropriate expert on a per-example basis?

In [1]:
import pandas as pd
import numpy as np

train_df = pd.read_csv("../data/processed/taxonomy_train.csv")
test_df = pd.read_csv("../data/processed/taxonomy_test.csv")

# ---------------------------------------------------------
# 1. Expected sizes
# ---------------------------------------------------------

assert len(train_df) == 1489
assert len(test_df) == 287
assert len(train_df) + len(test_df) == 1776

# ---------------------------------------------------------
# 2. Expected labels
# ---------------------------------------------------------

expected_train = {
    "workflow_error": 660,
    "constraint_error": 317,
    "tool_use_error": 237,
    "grounding_state_error": 244,
    "reasoning_value_error": 31,
}

expected_test = {
    "workflow_error": 138,
    "constraint_error": 70,
    "tool_use_error": 38,
    "grounding_state_error": 30,
    "reasoning_value_error": 11,
}

assert (
    train_df["failure_family"].value_counts().to_dict()
    == expected_train
)

assert (
    test_df["failure_family"].value_counts().to_dict()
    == expected_test
)

# ---------------------------------------------------------
# 3. Label mapping consistency
# ---------------------------------------------------------

label_map = {
    0: "workflow_error",
    1: "constraint_error",
    2: "tool_use_error",
    3: "grounding_state_error",
    4: "reasoning_value_error",
}

for df in [train_df, test_df]:

    reconstructed = df["family_label"].map(label_map)

    assert (
        reconstructed.values
        == df["failure_family"].values
    ).all()

# ---------------------------------------------------------
# 4. Group leakage
# ---------------------------------------------------------

train_groups = set(train_df["canonical_group"])
test_groups = set(test_df["canonical_group"])

overlap = train_groups & test_groups

print("Train groups:", len(train_groups))
print("Test groups:", len(test_groups))
print("Overlap:", len(overlap))

assert len(overlap) == 0

# ---------------------------------------------------------
# 5. Key uniqueness
# ---------------------------------------------------------

key_cols = [
    "dataset",
    "group_id",
    "message_index",
]

print(
    "Train duplicate keys:",
    train_df.duplicated(key_cols).sum()
)

print(
    "Test duplicate keys:",
    test_df.duplicated(key_cols).sum()
)

assert train_df.duplicated(key_cols).sum() == 0
assert test_df.duplicated(key_cols).sum() == 0

# ---------------------------------------------------------
# 6. Text integrity
# ---------------------------------------------------------

assert train_df["current_text"].notna().all()
assert test_df["current_text"].notna().all()

# Context may legitimately be empty.
print(
    "Empty train contexts:",
    train_df["context_text"].fillna("").eq("").sum()
)

print(
    "Empty test contexts:",
    test_df["context_text"].fillna("").eq("").sum()
)

# ---------------------------------------------------------
# Final
# ---------------------------------------------------------

print("\n✓ DATASET INTEGRITY CHECK PASSED")
print("Train:", train_df.shape)
print("Test:", test_df.shape)

Train groups: 335
Test groups: 84
Overlap: 0
Train duplicate keys: 0
Test duplicate keys: 0
Empty train contexts: 56
Empty test contexts: 18

✓ DATASET INTEGRITY CHECK PASSED
Train: (1489, 42)
Test: (287, 42)


In [2]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from sentence_transformers import SentenceTransformer

RANDOM_STATE = 42
N_CLASSES = 5

family_names = [
    "workflow_error",
    "constraint_error",
    "tool_use_error",
    "grounding_state_error",
    "reasoning_value_error",
]

y_train = train_df["family_label"].to_numpy()
y_test = test_df["family_label"].to_numpy()

groups_train = train_df["canonical_group"].to_numpy()
groups_test = test_df["canonical_group"].to_numpy()

In [3]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

reason_model = SentenceTransformer(
    MODEL_NAME
)

train_texts = (
    train_df["current_text"]
    .fillna("")
    .astype(str)
    .tolist()
)

test_texts = (
    test_df["current_text"]
    .fillna("")
    .astype(str)
    .tolist()
)

X_train_semantic = reason_model.encode(
    train_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

X_test_semantic = reason_model.encode(
    test_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

print(X_train_semantic.shape)
print(X_test_semantic.shape)

assert X_train_semantic.shape == (1489, 384)
assert X_test_semantic.shape == (287, 384)

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

(1489, 384)
(287, 384)


In [4]:
categorical_features = [
    "current_role",
    "current_tool",
    "previous_tool",
]

numeric_features = [
    "message_index",
    "is_tool_call",

    "current_char_length",
    "current_word_count",
    "context_char_length",
    "context_word_count",

    "previous_messages",
    "previous_tool_calls",
    "previous_assistant_messages",

    "parsed_tool_calls_in_context",
    "context_has_error_signal",
    "has_previous_tool_result",
    "has_previous_tool_call",

    "same_tool_as_previous",
    "current_tool_previous_count",
    "current_action_seen_before",
]

structured_features = (
    categorical_features
    + numeric_features
)

missing = [
    col
    for col in structured_features
    if col not in train_df.columns
]

print("Missing:", missing)
assert not missing

X_train_structured = (
    train_df[structured_features]
    .copy()
)

X_test_structured = (
    test_df[structured_features]
    .copy()
)

Missing: []


In [5]:
def make_semantic_expert():

    return LogisticRegression(
        max_iter=5000,
        class_weight=None,
        random_state=RANDOM_STATE,
    )


def make_structured_expert():

    numeric_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            ),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
    ])

    categorical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            ),
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=2,
            ),
        ),
    ])

    preprocessor = ColumnTransformer([
        (
            "num",
            numeric_pipeline,
            numeric_features,
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features,
        ),
    ])

    return Pipeline([
        (
            "preprocessor",
            preprocessor,
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=5000,
                class_weight=None,
                random_state=RANDOM_STATE,
            ),
        ),
    ])

In [6]:
cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

semantic_oof = np.zeros(
    (len(train_df), N_CLASSES)
)

structured_oof = np.zeros(
    (len(train_df), N_CLASSES)
)

In [7]:
for fold, (fit_idx, val_idx) in enumerate(
    cv.split(
        X_train_semantic,
        y_train,
        groups_train,
    ),
    start=1,
):

    print(
        f"Fold {fold}: "
        f"train={len(fit_idx)} "
        f"val={len(val_idx)}"
    )

    assert set(
        groups_train[fit_idx]
    ).isdisjoint(
        set(groups_train[val_idx])
    )

    # Semantic expert
    semantic_model = make_semantic_expert()

    semantic_model.fit(
        X_train_semantic[fit_idx],
        y_train[fit_idx],
    )

    semantic_oof[val_idx] = (
        semantic_model.predict_proba(
            X_train_semantic[val_idx]
        )
    )

    # Structured expert
    structured_model = make_structured_expert()

    structured_model.fit(
        X_train_structured.iloc[fit_idx],
        y_train[fit_idx],
    )

    structured_oof[val_idx] = (
        structured_model.predict_proba(
            X_train_structured.iloc[val_idx]
        )
    )

Fold 1: train=1190 val=299
Fold 2: train=1191 val=298
Fold 3: train=1191 val=298
Fold 4: train=1192 val=297
Fold 5: train=1192 val=297


In [8]:
assert np.allclose(
    semantic_oof.sum(axis=1),
    1.0,
)

assert np.allclose(
    structured_oof.sum(axis=1),
    1.0,
)

print("✓ OOF expert predictions ready")

✓ OOF expert predictions ready


In [9]:
semantic_oof_pred = semantic_oof.argmax(axis=1)
structured_oof_pred = structured_oof.argmax(axis=1)

semantic_correct = (
    semantic_oof_pred == y_train
)

structured_correct = (
    structured_oof_pred == y_train
)

In [10]:
gate_cases = pd.DataFrame({
    "semantic_correct": semantic_correct,
    "structured_correct": structured_correct,
})

print(
    pd.crosstab(
        gate_cases["semantic_correct"],
        gate_cases["structured_correct"],
    )
)

structured_correct  False  True 
semantic_correct                
False                 484    199
True                  214    592


In [11]:
gate_train_mask = (
    semantic_correct
    != structured_correct
)

print(
    "Useful gate examples:",
    gate_train_mask.sum()
)

print(
    "Percentage:",
    gate_train_mask.mean()
)

Useful gate examples: 413
Percentage: 0.277367360644728


In [12]:
gate_target = (
    semantic_correct[
        gate_train_mask
    ]
    .astype(int)
)

print(
    pd.Series(
        gate_target
    ).value_counts()
)

1    214
0    199
Name: count, dtype: int64


In [13]:
def probability_features(
    semantic_probs,
    structured_probs,
):

    semantic_top1 = (
        semantic_probs.max(axis=1)
    )

    structured_top1 = (
        structured_probs.max(axis=1)
    )

    semantic_sorted = np.sort(
        semantic_probs,
        axis=1,
    )

    structured_sorted = np.sort(
        structured_probs,
        axis=1,
    )

    semantic_margin = (
        semantic_sorted[:, -1]
        - semantic_sorted[:, -2]
    )

    structured_margin = (
        structured_sorted[:, -1]
        - structured_sorted[:, -2]
    )

    semantic_entropy = -np.sum(
        semantic_probs
        * np.log(
            semantic_probs + 1e-12
        ),
        axis=1,
    )

    structured_entropy = -np.sum(
        structured_probs
        * np.log(
            structured_probs + 1e-12
        ),
        axis=1,
    )

    disagreement = (
        semantic_probs.argmax(axis=1)
        !=
        structured_probs.argmax(axis=1)
    ).astype(int)

    features = np.column_stack([
        semantic_probs,
        structured_probs,

        semantic_top1,
        structured_top1,

        semantic_margin,
        structured_margin,

        semantic_entropy,
        structured_entropy,

        disagreement,
    ])

    return features

In [14]:
X_gate_all = probability_features(
    semantic_oof,
    structured_oof,
)

X_gate_train = X_gate_all[
    gate_train_mask
]

print(
    "Gate feature shape:",
    X_gate_train.shape
)

print(
    "Gate target shape:",
    gate_target.shape
)

Gate feature shape: (413, 17)
Gate target shape: (413,)


In [15]:
gate_model = LogisticRegression(
    max_iter=5000,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)

gate_model.fit(
    X_gate_train,
    gate_target,
)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",5000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default 

Why balanced here?

Because this is a different binary task from failure classification. If semantic-only correctness strongly outnumbers structured-only correctness, the gate could otherwise learn:

always trust semantic.

Balanced weighting is appropriate to test whether the routing signal is learnable.

In [16]:
final_semantic = make_semantic_expert()

final_semantic.fit(
    X_train_semantic,
    y_train,
)


final_structured = make_structured_expert()

final_structured.fit(
    X_train_structured,
    y_train,
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](5,)","[0,1,2,3,4]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](19,)","['current_role','current_tool','previous_tool',...,'same_tool_as_previous', 'current_tool_previous_count','current_action_seen_before']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,19
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropp

In [18]:
semantic_test_probs = (
    final_semantic.predict_proba(
        X_test_semantic
    )
)

structured_test_probs = (
    final_structured.predict_proba(
        X_test_structured
    )
)

In [19]:
X_gate_test = probability_features(
    semantic_test_probs,
    structured_test_probs,
)

gate_semantic_probability = (
    gate_model.predict_proba(
        X_gate_test
    )[:, 1]
)

In [20]:
pd.Series(
    gate_semantic_probability
).describe(
    percentiles=[
        .1,
        .25,
        .5,
        .75,
        .9,
    ]
)

count    287.000000
mean       0.495226
std        0.094594
min        0.132161
10%        0.386948
25%        0.429686
50%        0.509327
75%        0.569892
90%        0.607313
max        0.663441
dtype: float64

In [21]:
alpha = gate_semantic_probability[
    :, None
]

gated_probs = (
    alpha
    * semantic_test_probs
    +
    (1 - alpha)
    * structured_test_probs
)

gated_pred = gated_probs.argmax(
    axis=1
)

In [22]:
def evaluate_model(
    name,
    y_true,
    y_pred,
):

    print(
        "\n"
        + "=" * 80
    )

    print(name)

    print("=" * 80)

    print(
        classification_report(
            y_true,
            y_pred,
            target_names=family_names,
            digits=4,
            zero_division=0,
        )
    )

    result = {
        "model": name,

        "accuracy":
            accuracy_score(
                y_true,
                y_pred,
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                y_pred,
            ),

        "macro_f1":
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            ),

        "weighted_f1":
            f1_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0,
            ),
    }

    for key, value in result.items():

        if key != "model":
            print(
                f"{key}: {value:.4f}"
            )

    return result

In [23]:
semantic_pred = (
    semantic_test_probs.argmax(
        axis=1
    )
)

structured_pred = (
    structured_test_probs.argmax(
        axis=1
    )
)

semantic_result = evaluate_model(
    "Semantic",
    y_test,
    semantic_pred,
)

structured_result = evaluate_model(
    "Structured",
    y_test,
    structured_pred,
)

gated_result = evaluate_model(
    "Gated fusion",
    y_test,
    gated_pred,
)


Semantic
                       precision    recall  f1-score   support

       workflow_error     0.6230    0.5507    0.5846       138
     constraint_error     0.5584    0.6143    0.5850        70
       tool_use_error     0.3030    0.2632    0.2817        38
grounding_state_error     0.2708    0.4333    0.3333        30
reasoning_value_error     0.8571    0.5455    0.6667        11

             accuracy                         0.5157       287
            macro avg     0.5225    0.4814    0.4903       287
         weighted avg     0.5370    0.5157    0.5215       287

accuracy: 0.5157
balanced_accuracy: 0.4814
macro_f1: 0.4903
weighted_f1: 0.5215

Structured
                       precision    recall  f1-score   support

       workflow_error     0.5739    0.4783    0.5217       138
     constraint_error     0.4348    0.4286    0.4317        70
       tool_use_error     0.2955    0.3421    0.3171        38
grounding_state_error     0.2157    0.3667    0.2716        30
reasoning_va

In [24]:
results = pd.DataFrame([
    semantic_result,
    structured_result,
    gated_result,
])

results

,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,Semantic,0.515679,0.481391,0.490268,0.521487
1,Structured,0.439024,0.432212,0.434730,0.450732
2,Gated fusion,0.515679,0.487607,0.493137,0.517191


In [25]:
semantic_correct_test = (
    semantic_pred == y_test
)

structured_correct_test = (
    structured_pred == y_test
)

test_gate_mask = (
    semantic_correct_test
    !=
    structured_correct_test
)

true_gate_test = (
    semantic_correct_test[
        test_gate_mask
    ]
    .astype(int)
)

pred_gate_test = (
    gate_semantic_probability[
        test_gate_mask
    ]
    >= 0.5
).astype(int)

In [26]:
print(
    classification_report(
        true_gate_test,
        pred_gate_test,
        target_names=[
            "trust_structured",
            "trust_semantic",
        ],
        digits=4,
        zero_division=0,
    )
)

print(
    "Gate routing accuracy:",
    accuracy_score(
        true_gate_test,
        pred_gate_test,
    )
)

                  precision    recall  f1-score   support

trust_structured     0.3913    0.3103    0.3462        29
  trust_semantic     0.6491    0.7255    0.6852        51

        accuracy                         0.5750        80
       macro avg     0.5202    0.5179    0.5157        80
    weighted avg     0.5557    0.5750    0.5623        80

Gate routing accuracy: 0.575


In [ ]:
# Since semantic is the stronger branch, the simplest routing baseline is:
# always trust semantic


always_semantic_gate_accuracy = (
    true_gate_test.mean()
)

print(
    "Always-semantic routing accuracy:",
    always_semantic_gate_accuracy
)

print(
    "Learned gate routing accuracy:",
    accuracy_score(
        true_gate_test,
        pred_gate_test,
    )
)

Always-semantic routing accuracy: 0.6375
Learned gate routing accuracy: 0.575


Can routing improve if the gate sees not only model confidence, but also the semantic and trajectory characteristics of the example itself?

So the next experiment should be Experiment: Context-aware gating, not another generic classifier.

In [28]:
# ============================================================
# EXPERIMENT 12 — CONTEXT-AWARE GATING
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
)

In [29]:
gate_structured_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="most_frequent"
                    ),
                ),
                (
                    "onehot",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=False,
                    ),
                ),
            ]),
            categorical_features,
        ),
        (
            "numeric",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="median"
                    ),
                ),
                (
                    "scaler",
                    StandardScaler(),
                ),
            ]),
            numeric_features,
        ),
    ],
    remainder="drop",
)

X_gate_struct_train = (
    gate_structured_preprocessor
    .fit_transform(X_train_structured)
)

X_gate_struct_test = (
    gate_structured_preprocessor
    .transform(X_test_structured)
)

print("Gate structured train:", X_gate_struct_train.shape)
print("Gate structured test:", X_gate_struct_test.shape)

Gate structured train: (1489, 222)
Gate structured test: (287, 222)


In [30]:
X_gate_prob_train = probability_features(
    semantic_oof,
    structured_oof,
)

X_gate_prob_test = probability_features(
    semantic_test_probs,
    structured_test_probs,
)

print("Probability features:", X_gate_prob_train.shape)

Probability features: (1489, 17)


In [31]:
X_gate_full_train = np.hstack([
    X_gate_prob_train,
    X_train_semantic,
    X_gate_struct_train,
])

X_gate_full_test = np.hstack([
    X_gate_prob_test,
    X_test_semantic,
    X_gate_struct_test,
])

print("Full gate train:", X_gate_full_train.shape)
print("Full gate test:", X_gate_full_test.shape)

Full gate train: (1489, 623)
Full gate test: (287, 623)


In [32]:
semantic_oof_pred = semantic_oof.argmax(axis=1)
structured_oof_pred = structured_oof.argmax(axis=1)

semantic_correct = (
    semantic_oof_pred == y_train
)

structured_correct = (
    structured_oof_pred == y_train
)

gate_train_mask = (
    semantic_correct != structured_correct
)

gate_target = (
    semantic_correct[gate_train_mask]
    .astype(int)
)

X_gate_train_selected = (
    X_gate_full_train[
        gate_train_mask
    ]
)

print(
    "Gate training examples:",
    X_gate_train_selected.shape
)

print(
    pd.Series(gate_target)
    .value_counts()
)

Gate training examples: (413, 623)
1    214
0    199
Name: count, dtype: int64


In [ ]:
context_gate = LogisticRegression(
    max_iter=5000,
    class_weight="balanced",
    # I used C=0.3 intentionally because this gate has many more dimensions but only ~413 routing examples. Stronger regularization is sensible here.
    C=0.3,
    random_state=42,
)

context_gate.fit(
    X_gate_train_selected,
    gate_target,
)

,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",0.3
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",5000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default 

In [34]:
alpha_context = (
    context_gate.predict_proba(
        X_gate_full_test
    )[:, 1]
)

print(
    pd.Series(alpha_context)
    .describe(
        percentiles=[
            .1,
            .25,
            .5,
            .75,
            .9,
        ]
    )
)

count    287.000000
mean       0.541406
std        0.113983
min        0.231153
10%        0.400206
25%        0.476637
50%        0.528154
75%        0.605788
90%        0.706974
max        0.843923
dtype: float64


In [35]:
alpha = alpha_context[:, None]

context_gated_probs = (
    alpha
    * semantic_test_probs
    +
    (1 - alpha)
    * structured_test_probs
)

context_gated_pred = (
    context_gated_probs.argmax(
        axis=1
    )
)

In [36]:
print("=" * 80)
print("CONTEXT-AWARE GATED FUSION")
print("=" * 80)

print(
    classification_report(
        y_test,
        context_gated_pred,
        target_names=family_names,
        digits=4,
        zero_division=0,
    )
)

context_gate_results = {
    "accuracy":
        accuracy_score(
            y_test,
            context_gated_pred,
        ),

    "balanced_accuracy":
        balanced_accuracy_score(
            y_test,
            context_gated_pred,
        ),

    "macro_f1":
        f1_score(
            y_test,
            context_gated_pred,
            average="macro",
            zero_division=0,
        ),

    "weighted_f1":
        f1_score(
            y_test,
            context_gated_pred,
            average="weighted",
            zero_division=0,
        ),
}

context_gate_results

CONTEXT-AWARE GATED FUSION
                       precision    recall  f1-score   support

       workflow_error     0.6061    0.5797    0.5926       138
     constraint_error     0.5616    0.5857    0.5734        70
       tool_use_error     0.3438    0.2895    0.3143        38
grounding_state_error     0.3095    0.4333    0.3611        30
reasoning_value_error     0.7500    0.5455    0.6316        11

             accuracy                         0.5261       287
            macro avg     0.5142    0.4867    0.4946       287
         weighted avg     0.5350    0.5261    0.5284       287



{'accuracy': 0.5261324041811847,
 'balanced_accuracy': 0.48673719872804544,
 'macro_f1': 0.4945989877568825,
 'weighted_f1': 0.5283665394246726}

In [37]:
semantic_pred = (
    semantic_test_probs.argmax(
        axis=1
    )
)

structured_pred = (
    structured_test_probs.argmax(
        axis=1
    )
)

semantic_correct_test = (
    semantic_pred == y_test
)

structured_correct_test = (
    structured_pred == y_test
)

test_gate_mask = (
    semantic_correct_test
    != structured_correct_test
)

true_gate_test = (
    semantic_correct_test[
        test_gate_mask
    ]
    .astype(int)
)

pred_gate_test = (
    alpha_context[
        test_gate_mask
    ]
    >= 0.5
).astype(int)

In [38]:
print(
    classification_report(
        true_gate_test,
        pred_gate_test,
        target_names=[
            "trust_structured",
            "trust_semantic",
        ],
        digits=4,
        zero_division=0,
    )
)

learned_gate_accuracy = (
    accuracy_score(
        true_gate_test,
        pred_gate_test,
    )
)

always_semantic_accuracy = (
    true_gate_test.mean()
)

print(
    "Context-aware gate accuracy:",
    learned_gate_accuracy
)

print(
    "Always-semantic routing:",
    always_semantic_accuracy
)

print(
    "Delta:",
    learned_gate_accuracy
    - always_semantic_accuracy
)

                  precision    recall  f1-score   support

trust_structured     0.4000    0.3448    0.3704        29
  trust_semantic     0.6545    0.7059    0.6792        51

        accuracy                         0.5750        80
       macro avg     0.5273    0.5254    0.5248        80
    weighted avg     0.5623    0.5750    0.5673        80

Context-aware gate accuracy: 0.575
Always-semantic routing: 0.6375
Delta: -0.0625


In [39]:
comparison = pd.DataFrame([
    {
        "model": "semantic",
        "accuracy": accuracy_score(
            y_test,
            semantic_pred,
        ),
        "balanced_accuracy":
            balanced_accuracy_score(
                y_test,
                semantic_pred,
            ),
        "macro_f1":
            f1_score(
                y_test,
                semantic_pred,
                average="macro",
            ),
    },

    {
        "model": "probability_gate",
        "accuracy": 0.515679,
        "balanced_accuracy": 0.487607,
        "macro_f1": 0.493137,
    },

    {
        "model": "context_aware_gate",
        **{
            k: context_gate_results[k]
            for k in [
                "accuracy",
                "balanced_accuracy",
                "macro_f1",
            ]
        },
    },
])

comparison

,model,accuracy,balanced_accuracy,macro_f1
0,semantic,0.515679,0.481391,0.490268
1,probability_gate,0.515679,0.487607,0.493137
2,context_aware_gate,0.526132,0.486737,0.494599


This is a useful result. **Experiment 12 improves the final classifier a little, but it still does not solve the routing problem.**

Your comparison is:

| Model                  |   Accuracy | Balanced Acc. |   Macro F1 |
| ---------------------- | ---------: | ------------: | ---------: |
| Semantic               |     0.5157 |        0.4814 |     0.4903 |
| Probability gate       |     0.5157 |    **0.4876** |     0.4931 |
| **Context-aware gate** | **0.5261** |        0.4867 | **0.4946** |

So relative to the semantic baseline:

[
\Delta Accuracy = +0.0105
]

[
\Delta MacroF1 = +0.0043
]

and weighted F1 also improves:

[
0.5215 \rightarrow 0.5284
]

That is your best accuracy so far.

However, the routing diagnostic still says:

```text
Context-aware gate accuracy: 0.575
Always-semantic routing:     0.6375
```

So the gate is still **worse than simply choosing the semantic expert whenever the two experts disagree**.

That sounds contradictory at first, but it isn't. Your soft gate does not perform a hard expert choice. It blends probabilities:

[
P_{final}=
\alpha P_s+(1-\alpha)P_r
]

Therefore it can improve the final class even when its binary interpretation of “which expert is correct?” is wrong. A structured probability distribution may help nudge the semantic model away from a bad class without the structured expert itself being the correct hard prediction.

This is an important distinction:

> **Probability fusion can help even when expert routing accuracy is poor.**

Your gate distribution also became more expressive than Experiment 11:

```text
mean   0.541
std    0.114
min    0.231
max    0.844
```

So adding semantic + trajectory characteristics gave the gate stronger variation in (\alpha). It is no longer concentrated as tightly near 0.5.

But the main conclusion remains:

> The model still cannot reliably identify the 29 structured-only rescue cases.

For `trust_structured`, recall is only:

[
0.3448
]

So out of 29 cases where structure is uniquely right, the gate identifies only about 10.

Meanwhile, semantic routing recall is:

[
0.7059
]

which is much easier.

### What we now know

Experiment 10 showed an **oracle gap**:

[
51.6% \rightarrow 61.7%
]

Experiment 11 showed that model-confidence features alone could not route well.

Experiment 12 showed that adding actual semantic and trajectory features improves the *final blend* slightly, but **routing itself remains weak**.

That suggests the true routing signal may not be captured by the current static features. It may depend on richer relations such as:

* what a previous tool actually returned,
* whether the current claim contradicts that result,
* whether a prerequisite succeeded,
* whether the current action is valid given the prior state,
* explicit argument/value consistency,
* ordering of actions across the trajectory.

Those are more like **state-transition relationships** than aggregate features.

### Next experiment

Before building a full custom Transformer, I would do one more very clean experiment: **class-dependent fusion weights**.

Right now you have one scalar:

[
\alpha(x)
]

for the entire probability vector.

But your previous experiments repeatedly showed that structure helps some classes more than others. So instead learn:

[
\alpha_1(x),...,\alpha_5(x)
]

and combine per class:

[
P_k(y|x)=
\alpha_k(x)P_{semantic,k}
+
(1-\alpha_k(x))P_{structured,k}
]

This lets the model learn patterns like:

```text
workflow_error      → usually trust semantic
constraint_error    → mostly semantic
tool_use_error      → more structured influence
grounding_state     → more structured influence
reasoning_value     → mostly semantic
```

That is directly motivated by your class-level results and is a better next step than another generic MLP.

I would call the next section:

**Experiment 13 — Class-Conditional Gated Fusion**.


In [40]:
required_objects = [
    "semantic_oof",
    "structured_oof",
    "semantic_test_probs",
    "structured_test_probs",
    "X_gate_full_train",
    "X_gate_full_test",
    "y_train",
    "y_test",
]

for name in required_objects:
    assert name in globals(), f"Missing: {name}"

print("Semantic OOF:", semantic_oof.shape)
print("Structured OOF:", structured_oof.shape)

print("Semantic test:", semantic_test_probs.shape)
print("Structured test:", structured_test_probs.shape)

print("Gate train:", X_gate_full_train.shape)
print("Gate test:", X_gate_full_test.shape)

print("✓ Experiment 13 inputs ready")

Semantic OOF: (1489, 5)
Structured OOF: (1489, 5)
Semantic test: (287, 5)
Structured test: (287, 5)
Gate train: (1489, 623)
Gate test: (287, 623)
✓ Experiment 13 inputs ready


In [41]:
N_CLASSES = len(family_names)

y_train_onehot = np.eye(
    N_CLASSES
)[y_train]

print(y_train_onehot.shape)
print(y_train_onehot[:5])

(1489, 5)
[[0. 1. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 1. 0.]]


In [42]:
# 1 = semantic probability is better
# 0 = structured probability is better

semantic_class_error = np.abs(
    semantic_oof
    - y_train_onehot
)

structured_class_error = np.abs(
    structured_oof
    - y_train_onehot
)

In [43]:
class_gate_targets = (
    semantic_class_error
    <
    structured_class_error
).astype(int)

In [44]:
print(
    "Class gate targets:",
    class_gate_targets.shape
)

Class gate targets: (1489, 5)


In [45]:
class_gate_ties = np.isclose(
    semantic_class_error,
    structured_class_error,
    atol=1e-8,
)

for k, family in enumerate(family_names):

    usable = (
        ~class_gate_ties[:, k]
    )

    target = (
        class_gate_targets[
            usable,
            k,
        ]
    )

    print("\n", family)

    print(
        "Usable:",
        usable.sum()
    )

    print(
        "Semantic better:",
        (target == 1).sum()
    )

    print(
        "Structured better:",
        (target == 0).sum()
    )


 workflow_error
Usable: 1489
Semantic better: 646
Structured better: 843

 constraint_error
Usable: 1489
Semantic better: 596
Structured better: 893

 tool_use_error
Usable: 1489
Semantic better: 601
Structured better: 888

 grounding_state_error
Usable: 1489
Semantic better: 670
Structured better: 819

 reasoning_value_error
Usable: 1489
Semantic better: 530
Structured better: 959


In [46]:
from sklearn.linear_model import LogisticRegression

class_gate_models = []

for k, family in enumerate(
    family_names
):

    usable = (
        ~class_gate_ties[:, k]
    )

    X_k = X_gate_full_train[
        usable
    ]

    y_k = class_gate_targets[
        usable,
        k
    ]

    print("\n" + "=" * 70)
    print(f"GATE FOR: {family}")
    print("=" * 70)

    print(
        "Training rows:",
        len(y_k)
    )

    print(
        "Target distribution:"
    )

    print(
        pd.Series(y_k)
        .value_counts()
        .sort_index()
    )

    gate = LogisticRegression(
        max_iter=5000,
        class_weight="balanced",

        # Same regularization idea as Exp 12:
        # many features, relatively little routing data.
        C=0.3,

        random_state=42,
    )

    gate.fit(
        X_k,
        y_k,
    )

    class_gate_models.append(
        gate
    )


GATE FOR: workflow_error
Training rows: 1489
Target distribution:
0    843
1    646
Name: count, dtype: int64

GATE FOR: constraint_error
Training rows: 1489
Target distribution:
0    893
1    596
Name: count, dtype: int64

GATE FOR: tool_use_error
Training rows: 1489
Target distribution:
0    888
1    601
Name: count, dtype: int64

GATE FOR: grounding_state_error
Training rows: 1489
Target distribution:
0    819
1    670
Name: count, dtype: int64

GATE FOR: reasoning_value_error
Training rows: 1489
Target distribution:
0    959
1    530
Name: count, dtype: int64


In [47]:
assert len(class_gate_models) == 5

print("✓ Five class-specific gates trained")

✓ Five class-specific gates trained


In [48]:
alpha_class = np.zeros(
    (
        len(y_test),
        N_CLASSES,
    )
)

In [49]:
for k, gate in enumerate(
    class_gate_models
):

    alpha_class[:, k] = (
        gate.predict_proba(
            X_gate_full_test
        )[:, 1]
    )

In [50]:
print(
    "Alpha matrix:",
    alpha_class.shape
)

pd.DataFrame(
    alpha_class,
    columns=family_names,
).describe().T

Alpha matrix: (287, 5)


,count,mean,std,min,25%,50%,75%,max
workflow_error,287.0,0.531158,0.157293,0.085119,0.464122,0.545973,0.634233,0.811458
constraint_error,287.0,0.519186,0.235840,0.002113,0.354366,0.561089,0.719447,0.883656
tool_use_error,287.0,0.499312,0.255012,0.010329,0.295664,0.525245,0.698364,0.955989
grounding_state_error,287.0,0.550628,0.238061,0.057040,0.347247,0.594032,0.743139,0.946930
reasoning_value_error,287.0,0.444906,0.398723,0.000003,0.026030,0.341731,0.884502,0.996947


In [51]:
alpha_summary = pd.DataFrame({
    "failure_family": family_names,
    "mean_semantic_weight":
        alpha_class.mean(axis=0),

    "median_semantic_weight":
        np.median(
            alpha_class,
            axis=0,
        ),
})

alpha_summary[
    "mean_structured_weight"
] = (
    1
    - alpha_summary[
        "mean_semantic_weight"
    ]
)

alpha_summary

,failure_family,mean_semantic_weight,median_semantic_weight,mean_structured_weight
0,workflow_error,0.531158,0.545973,0.468842
1,constraint_error,0.519186,0.561089,0.480814
2,tool_use_error,0.499312,0.525245,0.500688
3,grounding_state_error,0.550628,0.594032,0.449372
4,reasoning_value_error,0.444906,0.341731,0.555094


In [52]:
class_gated_raw_probs = (
    alpha_class
    * semantic_test_probs
    +
    (
        1 - alpha_class
    )
    * structured_test_probs
)

In [53]:
class_gated_raw_probs.sum(
    axis=1
)[:10]

array([0.94241632, 0.98909576, 0.94154056, 0.93327359, 0.95336473,
       0.94699042, 0.87517675, 0.87895058, 0.92350445, 0.93901837])

In [54]:
class_gated_probs = (
    class_gated_raw_probs
    /
    class_gated_raw_probs.sum(
        axis=1,
        keepdims=True,
    )
)

In [55]:
assert np.allclose(
    class_gated_probs.sum(axis=1),
    1.0,
)

print(
    class_gated_probs.sum(
        axis=1
    )[:10]
)

[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [56]:
class_gated_pred = (
    class_gated_probs.argmax(
        axis=1
    )
)

In [57]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

print("=" * 80)
print("CLASS-CONDITIONAL GATED FUSION")
print("=" * 80)

print(
    classification_report(
        y_test,
        class_gated_pred,
        target_names=family_names,
        digits=4,
        zero_division=0,
    )
)

CLASS-CONDITIONAL GATED FUSION
                       precision    recall  f1-score   support

       workflow_error     0.6016    0.5580    0.5789       138
     constraint_error     0.5256    0.5857    0.5541        70
       tool_use_error     0.3438    0.2895    0.3143        38
grounding_state_error     0.3171    0.4333    0.3662        30
reasoning_value_error     0.7500    0.5455    0.6316        11

             accuracy                         0.5157       287
            macro avg     0.5076    0.4824    0.4890       287
         weighted avg     0.5249    0.5157    0.5176       287



In [58]:
class_gate_results = {
    "accuracy":
        accuracy_score(
            y_test,
            class_gated_pred,
        ),

    "balanced_accuracy":
        balanced_accuracy_score(
            y_test,
            class_gated_pred,
        ),

    "macro_f1":
        f1_score(
            y_test,
            class_gated_pred,
            average="macro",
            zero_division=0,
        ),

    "weighted_f1":
        f1_score(
            y_test,
            class_gated_pred,
            average="weighted",
            zero_division=0,
        ),
}

class_gate_results

{'accuracy': 0.5156794425087108,
 'balanced_accuracy': 0.4823893726410889,
 'macro_f1': 0.48901265344556666,
 'weighted_f1': 0.5176120616123923}

In [59]:
print(
    "\nConfusion matrix:"
)

print(
    confusion_matrix(
        y_test,
        class_gated_pred,
    )
)


Confusion matrix:
[[77 20 19 21  1]
 [25 41  2  2  0]
 [12 12 11  3  0]
 [11  5  0 13  1]
 [ 3  0  0  2  6]]


In [60]:
comparison = pd.DataFrame([
    {
        "model": "semantic",
        "accuracy": 0.515679,
        "balanced_accuracy": 0.481391,
        "macro_f1": 0.490268,
        "weighted_f1": 0.521487,
    },

    {
        "model": "probability_gate",
        "accuracy": 0.515679,
        "balanced_accuracy": 0.487607,
        "macro_f1": 0.493137,
        "weighted_f1": 0.517191,
    },

    {
        "model": "context_aware_gate",
        "accuracy": 0.526132,
        "balanced_accuracy": 0.486737,
        "macro_f1": 0.494599,
        "weighted_f1": 0.528367,
    },

    {
        "model": "class_conditional_gate",
        **class_gate_results,
    },
])

comparison

,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,semantic,0.515679,0.481391,0.490268,0.521487
1,probability_gate,0.515679,0.487607,0.493137,0.517191
2,context_aware_gate,0.526132,0.486737,0.494599,0.528367
3,class_conditional_gate,0.515679,0.482389,0.489013,0.517612


In [61]:
comparison.sort_values(
    "macro_f1",
    ascending=False,
)

,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
2,context_aware_gate,0.526132,0.486737,0.494599,0.528367
1,probability_gate,0.515679,0.487607,0.493137,0.517191
0,semantic,0.515679,0.481391,0.490268,0.521487
3,class_conditional_gate,0.515679,0.482389,0.489013,0.517612


In [62]:
semantic_macro = 0.490268

print(
    "Δ Macro F1 vs semantic:",
    class_gate_results[
        "macro_f1"
    ]
    - semantic_macro
)

print(
    "Δ Accuracy vs semantic:",
    class_gate_results[
        "accuracy"
    ]
    - 0.515679
)

print(
    "Δ Macro F1 vs context gate:",
    class_gate_results[
        "macro_f1"
    ]
    - 0.494599
)

Δ Macro F1 vs semantic: -0.001255346554433323
Δ Accuracy vs semantic: 4.4250871078510556e-07
Δ Macro F1 vs context gate: -0.005586346554433352


In [63]:
true_class_alpha = (
    alpha_class[
        np.arange(len(y_test)),
        y_test,
    ]
)

In [64]:
true_alpha_df = pd.DataFrame({
    "failure_family": [
        family_names[y]
        for y in y_test
    ],

    "semantic_weight":
        true_class_alpha,
})

In [65]:
true_alpha_summary = (
    true_alpha_df
    .groupby(
        "failure_family"
    )[
        "semantic_weight"
    ]
    .agg([
        "count",
        "mean",
        "median",
        "std",
        "min",
        "max",
    ])
)

true_alpha_summary

,count,mean,median,std,min,max
failure_family,,,,,,
constraint_error,70,0.595288,0.604406,0.151163,0.146429,0.862826
grounding_state_error,30,0.604836,0.716253,0.230005,0.153515,0.877265
reasoning_value_error,11,0.453424,0.491877,0.345230,0.096621,0.957316
tool_use_error,38,0.491534,0.521499,0.252847,0.030149,0.889039
workflow_error,138,0.501996,0.529483,0.151583,0.090690,0.811458


In [66]:
gate_behavior_df = test_df[
    [
        "current_role",
        "failure_family",
        "current_tool",
    ]
].copy()

gate_behavior_df[
    "true_class_semantic_weight"
] = true_class_alpha

In [67]:
gate_behavior_df.groupby(
    "current_role"
)[
    "true_class_semantic_weight"
].agg([
    "count",
    "mean",
    "median",
])

,count,mean,median
current_role,,,
ASSISTANT,137,0.567470,0.584295
TOOL_CALL,150,0.500089,0.521488


In [68]:
gate_behavior_df.groupby(
    [
        "current_role",
        "failure_family",
    ]
)[
    "true_class_semantic_weight"
].agg([
    "count",
    "mean",
    "median",
])

count      mean    median
current_role failure_family                                  
ASSISTANT    constraint_error          38  0.586398  0.606197
             grounding_state_error     22  0.595928  0.678901
             reasoning_value_error      5  0.699310  0.612661
             tool_use_error            14  0.384249  0.296528
             workflow_error            58  0.577133  0.568069
TOOL_CALL    constraint_error          32  0.605845  0.602567
             grounding_state_error      8  0.629331  0.745098
             reasoning_value_error      6  0.248518  0.116649
             tool_use_error            24  0.554117  0.553904
             workflow_error            80  0.447522  0.477579

In [69]:
# ============================================================
# ROLE-AWARE GATING ABLATION
# ============================================================

from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)

# ------------------------------------------------------------
# 1. Encode role only
# ------------------------------------------------------------

role_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False,
)

X_role_train = role_encoder.fit_transform(
    train_df[["current_role"]]
)

X_role_test = role_encoder.transform(
    test_df[["current_role"]]
)

print(
    role_encoder.get_feature_names_out()
)

print(
    X_role_train.shape,
    X_role_test.shape,
)

['current_role_ASSISTANT' 'current_role_TOOL_CALL']
(1489, 2) (287, 2)


In [70]:
X_role_gate_train_all = np.hstack([
    X_gate_prob_train,
    X_role_train,
])

X_role_gate_test = np.hstack([
    X_gate_prob_test,
    X_role_test,
])

print(
    X_role_gate_train_all.shape,
    X_role_gate_test.shape,
)

(1489, 19) (287, 19)


In [71]:
# exactly one expert correct
gate_train_mask = (
    semantic_correct
    != structured_correct
)

gate_target = (
    semantic_correct[
        gate_train_mask
    ]
    .astype(int)
)

X_role_gate_train = (
    X_role_gate_train_all[
        gate_train_mask
    ]
)

print(X_role_gate_train.shape)
print(
    pd.Series(gate_target)
    .value_counts()
)

(413, 19)
1    214
0    199
Name: count, dtype: int64


In [72]:
role_gate = LogisticRegression(
    max_iter=5000,
    class_weight="balanced",
    C=0.5,
    random_state=42,
)

role_gate.fit(
    X_role_gate_train,
    gate_target,
)

,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",0.5
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",5000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default 

In [73]:
alpha_role = (
    role_gate.predict_proba(
        X_role_gate_test
    )[:, 1]
)

pd.Series(
    alpha_role
).describe(
    percentiles=[
        .1,
        .25,
        .5,
        .75,
        .9,
    ]
)

count    287.000000
mean       0.499276
std        0.079131
min        0.232078
10%        0.393188
25%        0.453573
50%        0.509094
75%        0.555559
90%        0.597543
max        0.656950
dtype: float64

In [74]:
role_gated_probs = (
    alpha_role[:, None]
    * semantic_test_probs
    +
    (1 - alpha_role[:, None])
    * structured_test_probs
)

role_gated_pred = (
    role_gated_probs.argmax(
        axis=1
    )
)

In [75]:
role_gate_results = {
    "accuracy": accuracy_score(
        y_test,
        role_gated_pred,
    ),

    "balanced_accuracy":
        balanced_accuracy_score(
            y_test,
            role_gated_pred,
        ),

    "macro_f1": f1_score(
        y_test,
        role_gated_pred,
        average="macro",
        zero_division=0,
    ),

    "weighted_f1": f1_score(
        y_test,
        role_gated_pred,
        average="weighted",
        zero_division=0,
    ),
}

role_gate_results

{'accuracy': 0.519163763066202,
 'balanced_accuracy': 0.4904639068025795,
 'macro_f1': 0.49488795651574014,
 'weighted_f1': 0.5210863827878409}

In [76]:
semantic_correct_test = (
    semantic_pred == y_test
)

structured_correct_test = (
    structured_pred == y_test
)

test_route_mask = (
    semantic_correct_test
    != structured_correct_test
)

true_route = (
    semantic_correct_test[
        test_route_mask
    ]
    .astype(int)
)

pred_route = (
    alpha_role[
        test_route_mask
    ]
    >= 0.5
).astype(int)

role_router_accuracy = accuracy_score(
    true_route,
    pred_route,
)

always_semantic_accuracy = (
    true_route.mean()
)

print(
    "Role-aware routing:",
    role_router_accuracy
)

print(
    "Always semantic:",
    always_semantic_accuracy
)

print(
    "Delta:",
    role_router_accuracy
    - always_semantic_accuracy
)

Role-aware routing: 0.5875
Always semantic: 0.6375
Delta: -0.04999999999999993


In [77]:
role_analysis = pd.DataFrame({
    "current_role":
        test_df["current_role"].values,

    "alpha_semantic":
        alpha_role,

    "semantic_correct":
        semantic_correct_test,

    "structured_correct":
        structured_correct_test,
})

role_analysis.groupby(
    "current_role"
)["alpha_semantic"].agg([
    "count",
    "mean",
    "median",
    "std",
])

,count,mean,median,std
current_role,,,,
ASSISTANT,137,0.509518,0.526989,0.073133
TOOL_CALL,150,0.489922,0.481790,0.083383


In [78]:
# ============================================================
# ROUTING EXPERIMENT COMPARISON
# ============================================================

routing_results = pd.DataFrame([
    {
        "model": "semantic",
        "accuracy": accuracy_score(y_test, semantic_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, semantic_pred),
        "macro_f1": f1_score(
            y_test, semantic_pred,
            average="macro",
            zero_division=0
        ),
        "weighted_f1": f1_score(
            y_test, semantic_pred,
            average="weighted",
            zero_division=0
        ),
    },

    {
        "model": "probability_gate",
        "accuracy": 0.515679,
        "balanced_accuracy": 0.487607,
        "macro_f1": 0.493137,
        "weighted_f1": 0.517191,
    },

    {
        "model": "context_aware_gate",
        "accuracy": 0.526132,
        "balanced_accuracy": 0.486737,
        "macro_f1": 0.494599,
        "weighted_f1": 0.528367,
    },

    {
        "model": "class_conditional_gate",
        "accuracy": 0.515679,
        "balanced_accuracy": 0.482389,
        "macro_f1": 0.489013,
        "weighted_f1": 0.517612,
    },

    {
        "model": "role_aware_gate",
        **role_gate_results,
    },
])

routing_results

,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,semantic,0.515679,0.481391,0.490268,0.521487
1,probability_gate,0.515679,0.487607,0.493137,0.517191
2,context_aware_gate,0.526132,0.486737,0.494599,0.528367
3,class_conditional_gate,0.515679,0.482389,0.489013,0.517612
4,role_aware_gate,0.519164,0.490464,0.494888,0.521086


In [79]:
# ============================================================
# DELTA VS SEMANTIC BASELINE
# ============================================================

baseline = (
    routing_results
    .set_index("model")
    .loc["semantic"]
)

metric_cols = [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "weighted_f1",
]

routing_comparison = routing_results.copy()

for metric in metric_cols:
    routing_comparison[
        f"delta_{metric}"
    ] = (
        routing_comparison[metric]
        - baseline[metric]
    )

routing_comparison.sort_values(
    "macro_f1",
    ascending=False
)

,model,accuracy,balanced_accuracy,macro_f1,weighted_f1,delta_accuracy,delta_balanced_accuracy,delta_macro_f1,delta_weighted_f1
4,role_aware_gate,0.519164,0.490464,0.494888,0.521086,3.484321e-03,0.009073,0.004620,-0.000401
2,context_aware_gate,0.526132,0.486737,0.494599,0.528367,1.045256e-02,0.005346,0.004331,0.006880
1,probability_gate,0.515679,0.487607,0.493137,0.517191,-4.425087e-07,0.006216,0.002869,-0.004296
0,semantic,0.515679,0.481391,0.490268,0.521487,0.000000e+00,0.000000,0.000000,0.000000
3,class_conditional_gate,0.515679,0.482389,0.489013,0.517612,-4.425087e-07,0.000998,-0.001255,-0.003875


In [80]:
# ============================================================
# ROUTER QUALITY COMPARISON
# ============================================================

router_comparison = pd.DataFrame([
    {
        "router": "always_semantic",
        "routing_accuracy": 0.6375,
    },
    {
        "router": "probability_gate",
        "routing_accuracy": 0.5750,
    },
    {
        "router": "context_aware_gate",
        "routing_accuracy": 0.5750,
    },
    {
        "router": "role_aware_gate",
        "routing_accuracy": role_router_accuracy,
    },
])

baseline_routing = (
    router_comparison
    .set_index("router")
    .loc["always_semantic", "routing_accuracy"]
)

router_comparison["delta_vs_always_semantic"] = (
    router_comparison["routing_accuracy"]
    - baseline_routing
)

router_comparison.sort_values(
    "routing_accuracy",
    ascending=False
)

,router,routing_accuracy,delta_vs_always_semantic
0,always_semantic,0.6375,0.0000
3,role_aware_gate,0.5875,-0.0500
1,probability_gate,0.5750,-0.0625
2,context_aware_gate,0.5750,-0.0625


In [81]:
# ============================================================
# FINAL ROUTING SUMMARY
# ============================================================

summary = (
    routing_comparison[
        [
            "model",
            "accuracy",
            "balanced_accuracy",
            "macro_f1",
            "weighted_f1",
            "delta_accuracy",
            "delta_balanced_accuracy",
            "delta_macro_f1",
            "delta_weighted_f1",
        ]
    ]
    .sort_values(
        "macro_f1",
        ascending=False
    )
    .reset_index(drop=True)
)

print("=" * 100)
print("ROUTING / MIXTURE-OF-EXPERTS EXPERIMENT SUMMARY")
print("=" * 100)

display(
    summary.style.format({
        "accuracy": "{:.4f}",
        "balanced_accuracy": "{:.4f}",
        "macro_f1": "{:.4f}",
        "weighted_f1": "{:.4f}",
        "delta_accuracy": "{:+.4f}",
        "delta_balanced_accuracy": "{:+.4f}",
        "delta_macro_f1": "{:+.4f}",
        "delta_weighted_f1": "{:+.4f}",
    })
)

print("\nRouter quality:")
display(
    router_comparison.style.format({
        "routing_accuracy": "{:.4f}",
        "delta_vs_always_semantic": "{:+.4f}",
    })
)

ROUTING / MIXTURE-OF-EXPERTS EXPERIMENT SUMMARY


,model,accuracy,balanced_accuracy,macro_f1,weighted_f1,delta_accuracy,delta_balanced_accuracy,delta_macro_f1,delta_weighted_f1
0,role_aware_gate,0.5192,0.4905,0.4949,0.5211,+0.0035,+0.0091,+0.0046,-0.0004
1,context_aware_gate,0.5261,0.4867,0.4946,0.5284,+0.0105,+0.0053,+0.0043,+0.0069
2,probability_gate,0.5157,0.4876,0.4931,0.5172,-0.0000,+0.0062,+0.0029,-0.0043
3,semantic,0.5157,0.4814,0.4903,0.5215,+0.0000,+0.0000,+0.0000,+0.0000
4,class_conditional_gate,0.5157,0.4824,0.4890,0.5176,-0.0000,+0.0010,-0.0013,-0.0039



Router quality:


,router,routing_accuracy,delta_vs_always_semantic
0,always_semantic,0.6375,+0.0000
1,probability_gate,0.5750,-0.0625
2,context_aware_gate,0.5750,-0.0625
3,role_aware_gate,0.5875,-0.0500


These results give us a fairly clear conclusion about the routing/MoE direction.

### What the experiment shows

The **role-aware gate is the best router tested so far**, but it is still worse than simply trusting the semantic expert when the two experts disagree:

| Router             | Routing accuracy |
| ------------------ | ---------------: |
| Always semantic    |       **0.6375** |
| Role-aware         |       **0.5875** |
| Probability gate   |           0.5750 |
| Context-aware gate |           0.5750 |

So adding `current_role` was useful: routing accuracy improved from **57.5% → 58.75%**. That supports your hypothesis that whether the current message is an `ASSISTANT` response or `TOOL_CALL` contains information about expert reliability.

But it is **not enough to learn a better hard router** than the semantic default.

### Classification tells a more interesting story

Your ranking by Macro F1 is:

| Model               |   Accuracy | Balanced Acc. |   Macro F1 |  Δ Macro F1 |
| ------------------- | ---------: | ------------: | ---------: | ----------: |
| **Role-aware gate** |     0.5192 |    **0.4905** | **0.4949** | **+0.0046** |
| Context-aware gate  | **0.5261** |        0.4867 |     0.4946 |     +0.0043 |
| Probability gate    |     0.5157 |        0.4876 |     0.4931 |     +0.0029 |
| Semantic            |     0.5157 |        0.4814 |     0.4903 |           — |
| Class-conditional   |     0.5157 |        0.4824 |     0.4890 |     −0.0013 |

This is useful because **role-aware and context-aware routing optimize different things**.

The context-aware gate gives the highest overall accuracy and weighted F1:

```text
Accuracy      0.5261
Weighted F1   0.5284
```

The role-aware gate gives the highest:

```text
Balanced Accuracy   0.4905
Macro F1            0.4949
```

For your imbalanced 5-family failure taxonomy, I would treat the latter as particularly important. Macro F1 gives every failure family equal influence instead of letting `workflow_error` dominate the result.

### The role signal is real, but weak

Your semantic weights are:

```text
ASSISTANT    mean = 0.5095
TOOL_CALL   mean = 0.4899
```

That's only about a **0.020 difference**.

So the model has learned:

> slightly favor semantic information for assistant messages and slightly favor structured information for tool calls.

That's directionally sensible, but the effect is small. `current_role` is therefore a useful **conditioning variable**, not a strong routing variable by itself.

More importantly, it improved Balanced Accuracy by:

```text
0.490464 - 0.481391
= +0.009073
```

That's actually the largest Balanced Accuracy gain among your routing experiments.

---

## The bigger research conclusion

At this point I would **stop trying increasingly complicated versions of the same two-expert gate**.

You've tested:

```text
Semantic expert
        │
        ├──── Probability gate
        ├──── Context-aware gate
        ├──── Class-conditional gate
        └──── Role-aware gate
                 │
                 ▼
        best Macro F1 = 0.4949

Semantic alone = 0.4903
```

The improvements are consistently tiny.

More importantly, your earlier oracle experiment showed:

```text
Semantic accuracy       ≈ 0.516
Structured accuracy     ≈ 0.439
Oracle branch accuracy  ≈ 0.617
```

That's the fascinating part.

There is roughly **10 percentage points of theoretical complementarity**, but your learned routers cannot reliably identify it.

That tells us something deeper:

> **The problem is no longer simply how to combine semantic and structured predictions. The problem is identifying the trajectory state under which each expert becomes reliable.**

And `current_role` is only a crude state variable.

---

# What I would do next

Move from **message-level routing** to **trajectory-state-aware routing**.

Instead of asking:

> Is this an ASSISTANT or TOOL_CALL message?

ask:

> **What state of the trajectory are we currently in?**

For example, construct features like:

```text
trajectory position
relative trajectory position

current role
previous role
role transition

current tool
previous tool
tool transition

number of previous tool calls
number of previous tool results

consecutive tool-call run length
consecutive assistant run length

distance since last tool result
distance since last error

current tool seen before
number of previous uses of current tool

previous result contained error
number of previous failed results

tool → result → assistant
assistant → tool
tool → same tool
tool → different tool

context length
trajectory length
```

Then we can represent a row more like:

```text
Trajectory:
USER
 ↓
ASSISTANT
 ↓
SEARCH
 ↓
RESULT
 ↓
SEARCH
 ↓
RESULT
 ↓
SEARCH      ← current point

State:
role              = TOOL_CALL
position           = 0.71
previous_tool      = search
same_tool          = True
tool_run_length    = 3
tool_seen_before   = True
previous_error     = False
results_seen       = 2
```

That contains substantially more information than:

```python
current_role = "TOOL_CALL"
```

And it directly matches what your failure families describe.

`workflow_error`, `tool_use_error`, and `grounding_state_error` are fundamentally **trajectory-dependent concepts**.

## The next experiment

I'd make the next notebook something like:

```text
12_trajectory_state_routing.ipynb
```

The central hypothesis would be:

> **H12: Expert reliability depends on trajectory state rather than only semantic confidence, predicted class, or current message role.**

And importantly, don't immediately evaluate only final classification.

We should first ask whether these features solve the actual bottleneck:

```text
Can trajectory state predict
"structured expert correct, semantic expert wrong"
vs
"semantic expert correct, structured expert wrong"?
```

Your current routing benchmark is:

```text
Always semantic        63.75%
Role-aware router      58.75%
```

So the first success criterion for Experiment 12 is extremely clear:

> **Can trajectory-state routing exceed 63.75% routing accuracy on disagreement cases?**

If it cannot, we have strong evidence that the missing complementarity is not recoverable from these observable trajectory features with the current dataset/model.

If it **can**, then we have found the mechanism needed to exploit that ~61.7% oracle ceiling.

That's a much stronger next research question than adding another generic fusion model.


In [82]:
# ============================================================
# TRAJECTORY-STATE ROUTING
# ============================================================

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
)


def build_trajectory_state_features(df):
    """
    Build state features available at the current trajectory step.
    No target/failure-family information is used.
    """
    out = df.copy()

    # --------------------------------------------------------
    # Numeric safety
    # --------------------------------------------------------

    numeric_source_cols = [
        "message_index",
        "current_char_length",
        "current_word_count",
        "context_char_length",
        "context_word_count",
        "previous_messages",
        "previous_tool_calls",
        "previous_assistant_messages",
        "parsed_tool_calls_in_context",
        "current_tool_previous_count",
        "same_tool_as_previous",
        "current_action_seen_before",
        "context_has_error_signal",
        "has_previous_tool_result",
        "has_previous_tool_call",
        "is_tool_call",
    ]

    for col in numeric_source_cols:
        if col in out.columns:
            out[col] = (
                pd.to_numeric(
                    out[col],
                    errors="coerce"
                )
                .fillna(0)
            )

    # --------------------------------------------------------
    # Trajectory depth
    # --------------------------------------------------------

    out["trajectory_depth"] = (
        out["previous_messages"] + 1
    )

    out["log_trajectory_depth"] = np.log1p(
        out["trajectory_depth"]
    )

    out["log_previous_tool_calls"] = np.log1p(
        out["previous_tool_calls"]
    )

    out["log_current_tool_previous_count"] = np.log1p(
        out["current_tool_previous_count"]
    )

    # --------------------------------------------------------
    # History composition
    # --------------------------------------------------------

    denom = (
        out["previous_messages"]
        .clip(lower=1)
    )

    out["tool_call_fraction_so_far"] = (
        out["previous_tool_calls"]
        / denom
    )

    out["assistant_fraction_so_far"] = (
        out["previous_assistant_messages"]
        / denom
    )

    # --------------------------------------------------------
    # Tool repetition intensity
    # --------------------------------------------------------

    tool_denom = (
        out["previous_tool_calls"]
        .clip(lower=1)
    )

    out["current_tool_repeat_ratio"] = (
        out["current_tool_previous_count"]
        / tool_denom
    )

    # --------------------------------------------------------
    # Context-state features
    # --------------------------------------------------------

    out["has_context"] = (
        out["context_char_length"] > 0
    ).astype(int)

    out["context_to_current_length_ratio"] = (
        out["context_char_length"]
        /
        out["current_char_length"].clip(lower=1)
    )

    # Prevent extreme ratio values dominating scaling
    out["context_to_current_length_ratio"] = (
        out["context_to_current_length_ratio"]
        .clip(upper=50)
    )

    # --------------------------------------------------------
    # Categorical interaction/state features
    # --------------------------------------------------------

    out["role_tool_state"] = (
        out["current_role"].astype(str)
        + "|"
        + out["current_tool"].astype(str)
    )

    out["tool_transition"] = (
        out["previous_tool"].astype(str)
        + "->"
        + out["current_tool"].astype(str)
    )

    out["role_previous_tool_state"] = (
        out["current_role"].astype(str)
        + "|prev="
        + out["previous_tool"].astype(str)
    )

    # Coarse trajectory-depth bucket.
    # Fixed rules -> no test-data fitting.
    out["trajectory_depth_bucket"] = pd.cut(
        out["trajectory_depth"],
        bins=[
            -np.inf,
            3,
            10,
            25,
            50,
            np.inf,
        ],
        labels=[
            "very_early",
            "early",
            "middle",
            "late",
            "very_late",
        ],
    ).astype(str)

    # --------------------------------------------------------
    # Tool-state category
    # --------------------------------------------------------

    out["tool_repetition_state"] = np.select(
        [
            out["current_tool"].eq("NO_TOOL"),
            out["current_tool_previous_count"].eq(0),
            out["current_tool_previous_count"].between(1, 2),
            out["current_tool_previous_count"].ge(3),
        ],
        [
            "no_tool",
            "first_use",
            "repeated_1_2",
            "repeated_3_plus",
        ],
        default="other",
    )

    return out

In [83]:
trajectory_train = build_trajectory_state_features(
    train_df
)

trajectory_test = build_trajectory_state_features(
    test_df
)

print(
    trajectory_train.shape,
    trajectory_test.shape
)

(1489, 56) (287, 56)


In [84]:
trajectory_categorical_features = [
    "current_role",
    "current_tool",
    "previous_tool",

    "role_tool_state",
    "tool_transition",
    "role_previous_tool_state",

    "trajectory_depth_bucket",
    "tool_repetition_state",
]


trajectory_numeric_features = [
    # position / depth
    "message_index",
    "trajectory_depth",
    "log_trajectory_depth",

    # history amounts
    "previous_messages",
    "previous_tool_calls",
    "previous_assistant_messages",
    "log_previous_tool_calls",

    # repetition
    "same_tool_as_previous",
    "current_action_seen_before",
    "current_tool_previous_count",
    "log_current_tool_previous_count",
    "current_tool_repeat_ratio",

    # visible state
    "parsed_tool_calls_in_context",
    "context_has_error_signal",
    "has_previous_tool_result",
    "has_previous_tool_call",
    "has_context",

    # composition
    "tool_call_fraction_so_far",
    "assistant_fraction_so_far",

    # size / complexity
    "current_char_length",
    "current_word_count",
    "context_char_length",
    "context_word_count",
    "context_to_current_length_ratio",

    "is_tool_call",
]

In [85]:
trajectory_router_features = (
    trajectory_categorical_features
    + trajectory_numeric_features
)

missing = [
    col
    for col in trajectory_router_features
    if col not in trajectory_train.columns
]

print("Missing:", missing)

assert not missing

print(
    "Trajectory router features:",
    len(trajectory_router_features)
)

Missing: []
Trajectory router features: 33


In [86]:
print(
    X_gate_prob_train.shape,
    X_gate_prob_test.shape
)

(1489, 17) (287, 17)


In [87]:
trajectory_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="most_frequent"
                    ),
                ),
                (
                    "onehot",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=False,
                        min_frequency=2,
                    ),
                ),
            ]),
            trajectory_categorical_features,
        ),

        (
            "numeric",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="median"
                    ),
                ),
                (
                    "scaler",
                    StandardScaler(),
                ),
            ]),
            trajectory_numeric_features,
        ),
    ],
    remainder="drop",
)


X_trajectory_train = (
    trajectory_preprocessor
    .fit_transform(
        trajectory_train[
            trajectory_router_features
        ]
    )
)

X_trajectory_test = (
    trajectory_preprocessor
    .transform(
        trajectory_test[
            trajectory_router_features
        ]
    )
)

print(
    "Trajectory train:",
    X_trajectory_train.shape
)

print(
    "Trajectory test:",
    X_trajectory_test.shape
)

Trajectory train: (1489, 564)
Trajectory test: (287, 564)


In [88]:
X_state_router_all_train = np.hstack([
    X_gate_prob_train,
    X_trajectory_train,
])

X_state_router_test = np.hstack([
    X_gate_prob_test,
    X_trajectory_test,
])

print(
    "Router train:",
    X_state_router_all_train.shape
)

print(
    "Router test:",
    X_state_router_test.shape
)

Router train: (1489, 581)
Router test: (287, 581)


In [89]:
semantic_oof_pred = (
    semantic_oof.argmax(axis=1)
)

structured_oof_pred = (
    structured_oof.argmax(axis=1)
)


semantic_correct_train = (
    semantic_oof_pred == y_train
)

structured_correct_train = (
    structured_oof_pred == y_train
)


trajectory_route_mask = (
    semantic_correct_train
    !=
    structured_correct_train
)


trajectory_route_target = (
    semantic_correct_train[
        trajectory_route_mask
    ]
    .astype(int)
)


X_state_router_train = (
    X_state_router_all_train[
        trajectory_route_mask
    ]
)


print(
    "Routing training examples:",
    X_state_router_train.shape
)

print(
    pd.Series(
        trajectory_route_target
    ).value_counts()
)

Routing training examples: (413, 581)
1    214
0    199
Name: count, dtype: int64


In [90]:
trajectory_router = LogisticRegression(
    max_iter=5000,

    class_weight="balanced",

    # Stronger regularization than normal classifier
    C=0.1,

    random_state=42,
)

trajectory_router.fit(
    X_state_router_train,
    trajectory_route_target,
)

,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",0.1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",5000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default 

In [91]:
alpha_trajectory = (
    trajectory_router
    .predict_proba(
        X_state_router_test
    )[:, 1]
)

pd.Series(
    alpha_trajectory
).describe(
    percentiles=[
        .1,
        .25,
        .5,
        .75,
        .9,
    ]
)

count    287.000000
mean       0.530368
std        0.112476
min        0.225596
10%        0.389513
25%        0.449587
50%        0.531003
75%        0.607564
90%        0.684732
max        0.794587
dtype: float64

In [92]:
semantic_pred = (
    semantic_test_probs.argmax(
        axis=1
    )
)

structured_pred = (
    structured_test_probs.argmax(
        axis=1
    )
)


semantic_correct_test = (
    semantic_pred == y_test
)

structured_correct_test = (
    structured_pred == y_test
)


test_disagreement_mask = (
    semantic_correct_test
    !=
    structured_correct_test
)


true_router_test = (
    semantic_correct_test[
        test_disagreement_mask
    ]
    .astype(int)
)


pred_router_test = (
    alpha_trajectory[
        test_disagreement_mask
    ]
    >= 0.5
).astype(int)

In [93]:
print("=" * 80)
print("TRAJECTORY-STATE ROUTING QUALITY")
print("=" * 80)

print(
    classification_report(
        true_router_test,
        pred_router_test,
        target_names=[
            "trust_structured",
            "trust_semantic",
        ],
        digits=4,
        zero_division=0,
    )
)


trajectory_router_accuracy = (
    accuracy_score(
        true_router_test,
        pred_router_test,
    )
)


always_semantic_router_accuracy = (
    true_router_test.mean()
)


print(
    "Trajectory router:",
    trajectory_router_accuracy
)

print(
    "Always semantic:",
    always_semantic_router_accuracy
)

print(
    "Delta:",
    trajectory_router_accuracy
    - always_semantic_router_accuracy
)

TRAJECTORY-STATE ROUTING QUALITY
                  precision    recall  f1-score   support

trust_structured     0.4375    0.4828    0.4590        29
  trust_semantic     0.6875    0.6471    0.6667        51

        accuracy                         0.5875        80
       macro avg     0.5625    0.5649    0.5628        80
    weighted avg     0.5969    0.5875    0.5914        80

Trajectory router: 0.5875
Always semantic: 0.6375
Delta: -0.04999999999999993


In [94]:
hard_routed_pred = np.where(
    alpha_trajectory >= 0.5,
    semantic_pred,
    structured_pred,
)

In [95]:
hard_router_results = {
    "accuracy":
        accuracy_score(
            y_test,
            hard_routed_pred,
        ),

    "balanced_accuracy":
        balanced_accuracy_score(
            y_test,
            hard_routed_pred,
        ),

    "macro_f1":
        f1_score(
            y_test,
            hard_routed_pred,
            average="macro",
            zero_division=0,
        ),

    "weighted_f1":
        f1_score(
            y_test,
            hard_routed_pred,
            average="weighted",
            zero_division=0,
        ),
}

hard_router_results

{'accuracy': 0.5017421602787456,
 'balanced_accuracy': 0.49224444510485704,
 'macro_f1': 0.4842408470215438,
 'weighted_f1': 0.5130195283081561}

In [96]:
alpha = (
    alpha_trajectory[:, None]
)

trajectory_gated_probs = (
    alpha
    * semantic_test_probs
    +
    (1 - alpha)
    * structured_test_probs
)

trajectory_gated_pred = (
    trajectory_gated_probs.argmax(
        axis=1
    )
)

In [97]:
trajectory_gate_results = {
    "accuracy":
        accuracy_score(
            y_test,
            trajectory_gated_pred,
        ),

    "balanced_accuracy":
        balanced_accuracy_score(
            y_test,
            trajectory_gated_pred,
        ),

    "macro_f1":
        f1_score(
            y_test,
            trajectory_gated_pred,
            average="macro",
            zero_division=0,
        ),

    "weighted_f1":
        f1_score(
            y_test,
            trajectory_gated_pred,
            average="weighted",
            zero_division=0,
        ),
}

trajectory_gate_results

{'accuracy': 0.519163763066202,
 'balanced_accuracy': 0.4838386480034077,
 'macro_f1': 0.4887536952874344,
 'weighted_f1': 0.52322539792516}

In [98]:
print("=" * 80)
print("TRAJECTORY-STATE SOFT GATED FUSION")
print("=" * 80)

print(
    classification_report(
        y_test,
        trajectory_gated_pred,
        target_names=family_names,
        digits=4,
        zero_division=0,
    )
)

TRAJECTORY-STATE SOFT GATED FUSION
                       precision    recall  f1-score   support

       workflow_error     0.6240    0.5652    0.5932       138
     constraint_error     0.5325    0.5857    0.5578        70
       tool_use_error     0.3333    0.2895    0.3099        38
grounding_state_error     0.2955    0.4333    0.3514        30
reasoning_value_error     0.7500    0.5455    0.6316        11

             accuracy                         0.5192       287
            macro avg     0.5071    0.4838    0.4888       287
         weighted avg     0.5337    0.5192    0.5232       287



In [99]:
full_routing_comparison = pd.DataFrame([
    {
        "model": "semantic",
        "accuracy": 0.515679,
        "balanced_accuracy": 0.481391,
        "macro_f1": 0.490268,
        "weighted_f1": 0.521487,
    },

    {
        "model": "probability_gate",
        "accuracy": 0.515679,
        "balanced_accuracy": 0.487607,
        "macro_f1": 0.493137,
        "weighted_f1": 0.517191,
    },

    {
        "model": "context_aware_gate",
        "accuracy": 0.526132,
        "balanced_accuracy": 0.486737,
        "macro_f1": 0.494599,
        "weighted_f1": 0.528367,
    },

    {
        "model": "class_conditional_gate",
        "accuracy": 0.515679,
        "balanced_accuracy": 0.482389,
        "macro_f1": 0.489013,
        "weighted_f1": 0.517612,
    },

    {
        "model": "role_aware_gate",
        "accuracy": 0.519164,
        "balanced_accuracy": 0.490464,
        "macro_f1": 0.494888,
        "weighted_f1": 0.521086,
    },

    {
        "model": "trajectory_state_soft_gate",
        **trajectory_gate_results,
    },

    {
        "model": "trajectory_state_hard_router",
        **hard_router_results,
    },
])

In [100]:
baseline = (
    full_routing_comparison
    .set_index("model")
    .loc["semantic"]
)

for metric in [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "weighted_f1",
]:

    full_routing_comparison[
        f"delta_{metric}"
    ] = (
        full_routing_comparison[metric]
        - baseline[metric]
    )


full_routing_comparison.sort_values(
    "macro_f1",
    ascending=False,
)

,model,accuracy,balanced_accuracy,macro_f1,weighted_f1,delta_accuracy,delta_balanced_accuracy,delta_macro_f1,delta_weighted_f1
4,role_aware_gate,0.519164,0.490464,0.494888,0.521086,0.003485,0.009073,0.004620,-0.000401
2,context_aware_gate,0.526132,0.486737,0.494599,0.528367,0.010453,0.005346,0.004331,0.006880
1,probability_gate,0.515679,0.487607,0.493137,0.517191,0.000000,0.006216,0.002869,-0.004296
0,semantic,0.515679,0.481391,0.490268,0.521487,0.000000,0.000000,0.000000,0.000000
3,class_conditional_gate,0.515679,0.482389,0.489013,0.517612,0.000000,0.000998,-0.001255,-0.003875
5,trajectory_state_soft_gate,0.519164,0.483839,0.488754,0.523225,0.003485,0.002448,-0.001514,0.001738
6,trajectory_state_hard_router,0.501742,0.492244,0.484241,0.513020,-0.013937,0.010853,-0.006027,-0.008467


In [101]:
router_quality = pd.DataFrame([
    {
        "router":
            "always_semantic",

        "routing_accuracy":
            0.6375,
    },

    {
        "router":
            "probability_gate",

        "routing_accuracy":
            0.5750,
    },

    {
        "router":
            "context_aware_gate",

        "routing_accuracy":
            0.5750,
    },

    {
        "router":
            "role_aware_gate",

        "routing_accuracy":
            0.5875,
    },

    {
        "router":
            "trajectory_state_router",

        "routing_accuracy":
            trajectory_router_accuracy,
    },
])


router_quality[
    "delta_vs_always_semantic"
] = (
    router_quality[
        "routing_accuracy"
    ]
    - 0.6375
)


router_quality.sort_values(
    "routing_accuracy",
    ascending=False,
)

,router,routing_accuracy,delta_vs_always_semantic
0,always_semantic,0.6375,0.0000
3,role_aware_gate,0.5875,-0.0500
4,trajectory_state_router,0.5875,-0.0500
1,probability_gate,0.5750,-0.0625
2,context_aware_gate,0.5750,-0.0625


Default to semantic. Only override with structured when there is strong evidence that structure is better.

In [102]:
threshold_results = []

for threshold in np.linspace(
    0.05,
    0.50,
    46,
):

    # Default semantic.
    # Switch to structured only if semantic weight is very low.
    routed_pred = semantic_pred.copy()

    override_mask = (
        alpha_trajectory < threshold
    )

    routed_pred[
        override_mask
    ] = structured_pred[
        override_mask
    ]

    threshold_results.append({
        "threshold": threshold,
        "override_count": override_mask.sum(),

        "accuracy":
            accuracy_score(
                y_test,
                routed_pred,
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_test,
                routed_pred,
            ),

        "macro_f1":
            f1_score(
                y_test,
                routed_pred,
                average="macro",
                zero_division=0,
            ),

        "weighted_f1":
            f1_score(
                y_test,
                routed_pred,
                average="weighted",
                zero_division=0,
            ),
    })


threshold_df = pd.DataFrame(
    threshold_results
)

threshold_df.sort_values(
    "macro_f1",
    ascending=False,
).head(15)

,threshold,override_count,accuracy,balanced_accuracy,macro_f1,weighted_f1
23,0.28,3,0.519164,0.484248,0.492243,0.525557
25,0.30,6,0.519164,0.484248,0.492243,0.525557
24,0.29,4,0.519164,0.484248,0.492243,0.525557
11,0.16,0,0.515679,0.481391,0.490268,0.521487
1,0.06,0,0.515679,0.481391,0.490268,0.521487
17,0.22,0,0.515679,0.481391,0.490268,0.521487
16,0.21,0,0.515679,0.481391,0.490268,0.521487
15,0.20,0,0.515679,0.481391,0.490268,0.521487
14,0.19,0,0.515679,0.481391,0.490268,0.521487
13,0.18,0,0.515679,0.481391,0.490268,0.521487


In [103]:
semantic_oof_pred = semantic_oof.argmax(axis=1)
structured_oof_pred = structured_oof.argmax(axis=1)

# You need the trajectory router's OOF alpha values.

In [104]:
override_diagnostics = []

for threshold in np.linspace(
    0.05,
    0.50,
    46,
):

    override = (
        alpha_trajectory < threshold
    )

    if override.sum() == 0:
        continue

    semantic_wrong = (
        semantic_pred != y_test
    )

    structured_correct = (
        structured_pred == y_test
    )

    good_override = (
        override
        & semantic_wrong
        & structured_correct
    )

    bad_override = (
        override
        & (semantic_pred == y_test)
        & (structured_pred != y_test)
    )

    override_diagnostics.append({
        "threshold": threshold,
        "override_count": override.sum(),
        "good_overrides": good_override.sum(),
        "bad_overrides": bad_override.sum(),
        "override_precision":
            good_override.sum()
            / max(override.sum(), 1),
        "net_rescues":
            good_override.sum()
            - bad_override.sum(),
    })


override_df = pd.DataFrame(
    override_diagnostics
)

override_df.sort_values(
    "net_rescues",
    ascending=False,
).head(20)

,threshold,override_count,good_overrides,bad_overrides,override_precision,net_rescues
6,0.29,4,1,0,0.250000,1
7,0.30,6,1,0,0.166667,1
5,0.28,3,1,0,0.333333,1
0,0.23,1,0,0,0.000000,0
1,0.24,1,0,0,0.000000,0
8,0.31,7,1,1,0.142857,0
9,0.32,8,1,1,0.125000,0
4,0.27,1,0,0,0.000000,0
3,0.26,1,0,0,0.000000,0
2,0.25,1,0,0,0.000000,0


In [105]:
semantic_conf = (
    semantic_oof.max(axis=1)
)

structured_conf = (
    structured_oof.max(axis=1)
)

confidence_gap_train = (
    semantic_conf
    - structured_conf
)

l1_disagreement_train = (
    np.abs(
        semantic_oof
        - structured_oof
    )
    .sum(axis=1)
)

js_like_train = (
    np.square(
        semantic_oof
        - structured_oof
    )
    .sum(axis=1)
)

In [106]:
confidence_gap_test = (
    semantic_test_probs.max(axis=1)
    - structured_test_probs.max(axis=1)
)

l1_disagreement_test = (
    np.abs(
        semantic_test_probs
        - structured_test_probs
    )
    .sum(axis=1)
)

js_like_test = (
    np.square(
        semantic_test_probs
        - structured_test_probs
    )
    .sum(axis=1)
)

In [107]:
X_disagreement_train = np.column_stack([
    X_gate_prob_train,
    confidence_gap_train,
    l1_disagreement_train,
    js_like_train,
])

X_disagreement_test = np.column_stack([
    X_gate_prob_test,
    confidence_gap_test,
    l1_disagreement_test,
    js_like_test,
])

In [108]:
from sklearn.metrics import log_loss

print(
    "Semantic log loss:",
    log_loss(
        y_test,
        semantic_test_probs,
    )
)

print(
    "Structured log loss:",
    log_loss(
        y_test,
        structured_test_probs,
    )
)

Semantic log loss: 1.1403669118881226
Structured log loss: 1.1860002341679754


In [109]:
semantic_brier = np.mean(
    (
        semantic_test_probs
        - np.eye(5)[y_test]
    ) ** 2
)

structured_brier = np.mean(
    (
        structured_test_probs
        - np.eye(5)[y_test]
    ) ** 2
)

print(
    "Semantic multiclass Brier:",
    semantic_brier
)

print(
    "Structured multiclass Brier:",
    structured_brier
)

Semantic multiclass Brier: 0.12224764125926031
Structured multiclass Brier: 0.1256167855594516


In [110]:
rescue_target = (
    (semantic_oof_pred != y_train)
    &
    (structured_oof_pred == y_train)
).astype(int)

print(
    pd.Series(
        rescue_target
    ).value_counts()
)

0    1290
1     199
Name: count, dtype: int64


In [111]:
rescue_router = LogisticRegression(
    max_iter=5000,
    class_weight="balanced",
    C=0.1,
    random_state=42,
)

rescue_router.fit(
    X_state_router_all_train,
    rescue_target,
)

,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",0.1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",5000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default 

In [112]:
rescue_probability = (
    rescue_router.predict_proba(
        X_state_router_test
    )[:, 1]
)

In [114]:
rescue_target = (
    (semantic_oof_pred != y_train)
    &
    (structured_oof_pred == y_train)
).astype(int)

In [115]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression

rescue_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

rescue_oof_probability = np.zeros(
    len(y_train),
    dtype=float,
)

for fold, (fit_idx, val_idx) in enumerate(
    rescue_cv.split(
        X_state_router_all_train,
        rescue_target,
        groups_train,
    ),
    start=1,
):
    model = LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        C=0.1,
        random_state=42,
    )

    model.fit(
        X_state_router_all_train[fit_idx],
        rescue_target[fit_idx],
    )

    rescue_oof_probability[val_idx] = (
        model.predict_proba(
            X_state_router_all_train[val_idx]
        )[:, 1]
    )

    print(
        f"Fold {fold}:",
        len(fit_idx),
        len(val_idx),
    )

print(
    pd.Series(
        rescue_oof_probability
    ).describe()
)

Fold 1: 1190 299
Fold 2: 1192 297
Fold 3: 1192 297
Fold 4: 1190 299
Fold 5: 1192 297
count    1489.000000
mean        0.304151
std         0.302420
min         0.004527
25%         0.061971
50%         0.107515
75%         0.648266
max         0.919118
dtype: float64


In [116]:
threshold_rows = []

for threshold in np.linspace(
    0.05,
    0.95,
    91,
):
    pred = semantic_oof_pred.copy()

    override = (
        rescue_oof_probability
        >= threshold
    )

    pred[override] = (
        structured_oof_pred[override]
    )

    good_override = (
        override
        &
        (semantic_oof_pred != y_train)
        &
        (structured_oof_pred == y_train)
    )

    bad_override = (
        override
        &
        (semantic_oof_pred == y_train)
        &
        (structured_oof_pred != y_train)
    )

    threshold_rows.append({
        "threshold": threshold,

        "override_count":
            override.sum(),

        "good_overrides":
            good_override.sum(),

        "bad_overrides":
            bad_override.sum(),

        "net_rescues":
            good_override.sum()
            - bad_override.sum(),

        "accuracy":
            accuracy_score(
                y_train,
                pred,
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_train,
                pred,
            ),

        "macro_f1":
            f1_score(
                y_train,
                pred,
                average="macro",
                zero_division=0,
            ),

        "weighted_f1":
            f1_score(
                y_train,
                pred,
                average="weighted",
                zero_division=0,
            ),
    })

oof_threshold_df = pd.DataFrame(
    threshold_rows
)

In [117]:
oof_threshold_df.sort_values(
    "macro_f1",
    ascending=False,
).head(20)

,threshold,override_count,good_overrides,bad_overrides,net_rescues,accuracy,balanced_accuracy,macro_f1,weighted_f1
68,0.73,201,89,67,22,0.556078,0.492447,0.522049,0.549348
65,0.70,251,105,85,20,0.554735,0.490365,0.521155,0.547707
67,0.72,215,90,74,16,0.552048,0.488375,0.518300,0.545002
66,0.71,237,98,83,15,0.551377,0.487441,0.518052,0.544149
64,0.69,270,109,97,12,0.549362,0.486908,0.516680,0.542531
63,0.68,298,117,109,8,0.546676,0.481425,0.512051,0.538678
54,0.59,456,168,167,1,0.541974,0.476343,0.507620,0.533161
62,0.67,328,121,123,-2,0.539960,0.476049,0.506656,0.532012
61,0.66,353,130,131,-1,0.540631,0.475508,0.506297,0.532071
51,0.56,485,178,179,-1,0.540631,0.475547,0.505914,0.531750


In [118]:
best_row = (
    oof_threshold_df
    .sort_values(
        "macro_f1",
        ascending=False,
    )
    .iloc[0]
)

BEST_RESCUE_THRESHOLD = float(
    best_row["threshold"]
)

print(
    "Selected threshold:",
    BEST_RESCUE_THRESHOLD
)

print(best_row)

Selected threshold: 0.73
threshold              0.730000
override_count       201.000000
good_overrides        89.000000
bad_overrides         67.000000
net_rescues           22.000000
accuracy               0.556078
balanced_accuracy      0.492447
macro_f1               0.522049
weighted_f1            0.549348
Name: 68, dtype: float64


In [119]:
final_rescue_router = LogisticRegression(
    max_iter=5000,
    class_weight="balanced",
    C=0.1,
    random_state=42,
)

final_rescue_router.fit(
    X_state_router_all_train,
    rescue_target,
)

,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",0.1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",5000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default 

In [120]:
rescue_probability_test = (
    final_rescue_router
    .predict_proba(
        X_state_router_test
    )[:, 1]
)

In [121]:
rescue_override = (
    rescue_probability_test
    >= BEST_RESCUE_THRESHOLD
)

selective_pred = semantic_pred.copy()

selective_pred[
    rescue_override
] = structured_pred[
    rescue_override
]

In [122]:
selective_results = {
    "accuracy":
        accuracy_score(
            y_test,
            selective_pred,
        ),

    "balanced_accuracy":
        balanced_accuracy_score(
            y_test,
            selective_pred,
        ),

    "macro_f1":
        f1_score(
            y_test,
            selective_pred,
            average="macro",
            zero_division=0,
        ),

    "weighted_f1":
        f1_score(
            y_test,
            selective_pred,
            average="weighted",
            zero_division=0,
        ),
}

selective_results

{'accuracy': 0.5226480836236934,
 'balanced_accuracy': 0.4828819083281326,
 'macro_f1': 0.48695807680686853,
 'weighted_f1': 0.5282652101338005}

In [123]:
print(
    classification_report(
        y_test,
        selective_pred,
        target_names=family_names,
        digits=4,
        zero_division=0,
    )
)

                       precision    recall  f1-score   support

       workflow_error     0.6320    0.5725    0.6008       138
     constraint_error     0.5676    0.6000    0.5833        70
       tool_use_error     0.3333    0.2632    0.2941        38
grounding_state_error     0.2600    0.4333    0.3250        30
reasoning_value_error     0.7500    0.5455    0.6316        11

             accuracy                         0.5226       287
            macro avg     0.5086    0.4829    0.4870       287
         weighted avg     0.5424    0.5226    0.5283       287



In [124]:
good_test_override = (
    rescue_override
    &
    (semantic_pred != y_test)
    &
    (structured_pred == y_test)
)

bad_test_override = (
    rescue_override
    &
    (semantic_pred == y_test)
    &
    (structured_pred != y_test)
)

print(
    "Selected threshold:",
    BEST_RESCUE_THRESHOLD
)

print(
    "Overrides:",
    rescue_override.sum()
)

print(
    "Good overrides:",
    good_test_override.sum()
)

print(
    "Bad overrides:",
    bad_test_override.sum()
)

print(
    "Net rescues:",
    good_test_override.sum()
    - bad_test_override.sum()
)

Selected threshold: 0.73
Overrides: 34
Good overrides: 14
Bad overrides: 12
Net rescues: 2


In [125]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# Class weights based only on TRAIN
# ---------------------------------------------------------

classes = np.unique(y_train)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train,
)

class_weight_map = dict(
    zip(classes, class_weights)
)

print("Class weights:")
for cls, weight in class_weight_map.items():
    print(
        label_map[cls],
        round(weight, 3),
    )

Class weights:
workflow_error 0.451
constraint_error 0.939
tool_use_error 1.257
grounding_state_error 1.22
reasoning_value_error 9.606


In [126]:
# ---------------------------------------------------------
# Base weight = importance of the TRUE failure class
# ---------------------------------------------------------

rescue_sample_weight = np.array([
    class_weight_map[y]
    for y in y_train
])

pd.DataFrame({
    "class": [
        label_map[y]
        for y in y_train
    ],
    "weight": rescue_sample_weight,
}).groupby("class")["weight"].first()

class
constraint_error         0.939432
grounding_state_error    1.220492
reasoning_value_error    9.606452
tool_use_error           1.256540
workflow_error           0.451212
Name: weight, dtype: float64

In [127]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression

weighted_rescue_oof = np.zeros(
    len(y_train),
    dtype=float,
)

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

for fold, (fit_idx, val_idx) in enumerate(
    cv.split(
        X_state_router_all_train,
        rescue_target,
        groups_train,
    ),
    start=1,
):

    router = LogisticRegression(
        max_iter=5000,
        C=0.1,
        random_state=42,
    )

    router.fit(
        X_state_router_all_train[fit_idx],
        rescue_target[fit_idx],
        sample_weight=rescue_sample_weight[fit_idx],
    )

    weighted_rescue_oof[val_idx] = (
        router.predict_proba(
            X_state_router_all_train[val_idx]
        )[:, 1]
    )

print(
    pd.Series(
        weighted_rescue_oof
    ).describe()
)

count    1489.000000
mean        0.117815
std         0.130159
min         0.003996
25%         0.023711
50%         0.044216
75%         0.202333
max         0.777104
dtype: float64


In [128]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)

rows = []

for threshold in np.linspace(
    0.05,
    0.95,
    91,
):

    pred = semantic_oof_pred.copy()

    override = (
        weighted_rescue_oof >= threshold
    )

    pred[override] = (
        structured_oof_pred[override]
    )

    good = (
        override
        & (semantic_oof_pred != y_train)
        & (structured_oof_pred == y_train)
    )

    bad = (
        override
        & (semantic_oof_pred == y_train)
        & (structured_oof_pred != y_train)
    )

    rows.append({
        "threshold": threshold,
        "override_count": override.sum(),
        "good_overrides": good.sum(),
        "bad_overrides": bad.sum(),
        "net_rescues": good.sum() - bad.sum(),

        "accuracy":
            accuracy_score(y_train, pred),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_train,
                pred,
            ),

        "macro_f1":
            f1_score(
                y_train,
                pred,
                average="macro",
                zero_division=0,
            ),

        "weighted_f1":
            f1_score(
                y_train,
                pred,
                average="weighted",
                zero_division=0,
            ),
    })

weighted_threshold_df = pd.DataFrame(rows)

display(
    weighted_threshold_df
    .sort_values(
        "macro_f1",
        ascending=False,
    )
    .head(15)
)

,threshold,override_count,good_overrides,bad_overrides,net_rescues,accuracy,balanced_accuracy,macro_f1,weighted_f1
29,0.34,115,58,30,28,0.560107,0.501972,0.527006,0.555962
28,0.33,125,61,34,27,0.559436,0.500824,0.526102,0.555123
30,0.35,107,53,29,24,0.557421,0.499776,0.524931,0.553213
26,0.31,146,68,44,24,0.557421,0.499071,0.524515,0.553113
27,0.32,136,64,40,24,0.557421,0.498743,0.524375,0.552967
23,0.28,184,80,60,20,0.554735,0.497858,0.523020,0.550794
25,0.30,152,69,48,21,0.555406,0.497317,0.522891,0.551023
31,0.36,97,46,27,19,0.554063,0.497064,0.522333,0.549874
24,0.29,166,73,53,20,0.554735,0.496662,0.522316,0.550380
33,0.38,79,39,20,19,0.554063,0.496195,0.522160,0.549419


In [129]:
best_weighted_row = (
    weighted_threshold_df
    .sort_values(
        "macro_f1",
        ascending=False,
    )
    .iloc[0]
)

WEIGHTED_RESCUE_THRESHOLD = float(
    best_weighted_row["threshold"]
)

print(
    "Weighted rescue threshold:",
    WEIGHTED_RESCUE_THRESHOLD
)

print(best_weighted_row)

Weighted rescue threshold: 0.33999999999999997
threshold              0.340000
override_count       115.000000
good_overrides        58.000000
bad_overrides         30.000000
net_rescues           28.000000
accuracy               0.560107
balanced_accuracy      0.501972
macro_f1               0.527006
weighted_f1            0.555962
Name: 29, dtype: float64


In [130]:
# ============================================================
# FINAL CLASS-WEIGHTED SELECTIVE RESCUE EVALUATION
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Freeze threshold selected entirely from TRAIN OOF
# ------------------------------------------------------------

WEIGHTED_RESCUE_THRESHOLD = 0.34


# ------------------------------------------------------------
# 2. Fit final weighted rescue router on all training data
# ------------------------------------------------------------

final_weighted_rescue_router = LogisticRegression(
    max_iter=5000,
    C=0.1,
    random_state=42,
)

final_weighted_rescue_router.fit(
    X_state_router_all_train,
    rescue_target,
    sample_weight=rescue_sample_weight,
)


# ------------------------------------------------------------
# 3. Predict rescue probabilities on untouched TEST
# ------------------------------------------------------------

weighted_rescue_probability_test = (
    final_weighted_rescue_router
    .predict_proba(X_state_router_test)[:, 1]
)

print("Test rescue probabilities:")
print(
    pd.Series(
        weighted_rescue_probability_test
    ).describe()
)


# ------------------------------------------------------------
# 4. Apply frozen selective override
# ------------------------------------------------------------

weighted_override = (
    weighted_rescue_probability_test
    >= WEIGHTED_RESCUE_THRESHOLD
)

weighted_selective_pred = semantic_pred.copy()

weighted_selective_pred[
    weighted_override
] = structured_pred[
    weighted_override
]


# ------------------------------------------------------------
# 5. Evaluate final predictions
# ------------------------------------------------------------

weighted_selective_results = {
    "accuracy": accuracy_score(
        y_test,
        weighted_selective_pred,
    ),

    "balanced_accuracy": balanced_accuracy_score(
        y_test,
        weighted_selective_pred,
    ),

    "macro_f1": f1_score(
        y_test,
        weighted_selective_pred,
        average="macro",
        zero_division=0,
    ),

    "weighted_f1": f1_score(
        y_test,
        weighted_selective_pred,
        average="weighted",
        zero_division=0,
    ),
}

print("\n" + "=" * 80)
print("CLASS-WEIGHTED SELECTIVE RESCUE")
print("=" * 80)

print(
    classification_report(
        y_test,
        weighted_selective_pred,
        target_names=family_names,
        digits=4,
        zero_division=0,
    )
)

print(weighted_selective_results)

print("\nConfusion matrix:")
print(
    confusion_matrix(
        y_test,
        weighted_selective_pred,
    )
)


# ------------------------------------------------------------
# 6. Analyze actual routing decisions
# ------------------------------------------------------------

good_weighted_override = (
    weighted_override
    & (semantic_pred != y_test)
    & (structured_pred == y_test)
)

bad_weighted_override = (
    weighted_override
    & (semantic_pred == y_test)
    & (structured_pred != y_test)
)

neutral_weighted_override = (
    weighted_override
    & ~good_weighted_override
    & ~bad_weighted_override
)

print("\n" + "=" * 80)
print("OVERRIDE ANALYSIS")
print("=" * 80)

print(
    "Threshold:",
    WEIGHTED_RESCUE_THRESHOLD,
)

print(
    "Overrides:",
    weighted_override.sum(),
)

print(
    "Good overrides:",
    good_weighted_override.sum(),
)

print(
    "Bad overrides:",
    bad_weighted_override.sum(),
)

print(
    "Neutral overrides:",
    neutral_weighted_override.sum(),
)

print(
    "Net rescues:",
    good_weighted_override.sum()
    - bad_weighted_override.sum(),
)

if weighted_override.sum() > 0:
    print(
        "Override precision:",
        good_weighted_override.sum()
        / weighted_override.sum(),
    )

Test rescue probabilities:
count    287.000000
mean       0.115840
std        0.126397
min        0.006958
25%        0.022191
50%        0.039422
75%        0.194083
max        0.622885
dtype: float64

CLASS-WEIGHTED SELECTIVE RESCUE
                       precision    recall  f1-score   support

       workflow_error     0.6239    0.4928    0.5506       138
     constraint_error     0.5696    0.6429    0.6040        70
       tool_use_error     0.2973    0.2895    0.2933        38
grounding_state_error     0.2407    0.4333    0.3095        30
reasoning_value_error     0.7500    0.5455    0.6316        11

             accuracy                         0.4983       287
            macro avg     0.4963    0.4808    0.4778       287
         weighted avg     0.5322    0.4983    0.5075       287

{'accuracy': 0.49825783972125437, 'balanced_accuracy': 0.48077446580879074, 'macro_f1': 0.47781404466250815, 'weighted_f1': 0.5074753806135214}

Confusion matrix:
[[68 18 22 29  1]
 [20 45  2  3 

In [132]:
# ============================================================
# FINAL ROUTING COMPARISON
# ============================================================

import pandas as pd

comparison = pd.DataFrame([
    {
        "model": "semantic",
        "accuracy": 0.5156794425087108,
        "balanced_accuracy": 0.481391,
        "macro_f1": 0.490268,
        "weighted_f1": 0.521487,
    },
    {
        "model": "probability_gate",
        "accuracy": 0.515679,
        "balanced_accuracy": 0.487607,
        "macro_f1": 0.493137,
        "weighted_f1": 0.517191,
    },
    {
        "model": "context_aware_gate",
        "accuracy": 0.526132,
        "balanced_accuracy": 0.486737,
        "macro_f1": 0.494599,
        "weighted_f1": 0.528367,
    },
    {
        "model": "role_aware_gate",
        "accuracy": 0.519164,
        "balanced_accuracy": 0.490464,
        "macro_f1": 0.494888,
        "weighted_f1": 0.521086,
    },
    {
        "model": "trajectory_state_soft_gate",
        "accuracy": 0.519164,
        "balanced_accuracy": 0.483839,
        "macro_f1": 0.488754,
        "weighted_f1": 0.523225,
    },
    {
        "model": "weighted_selective_rescue",
        **weighted_selective_results,
    },
])

# ------------------------------------------------------------
# Calculate deltas against semantic baseline
# ------------------------------------------------------------

baseline = comparison.loc[
    comparison["model"] == "semantic"
].iloc[0]

for metric in [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "weighted_f1",
]:
    comparison[f"delta_{metric}"] = (
        comparison[metric]
        - baseline[metric]
    )

# ------------------------------------------------------------
# Sort by primary metric
# ------------------------------------------------------------

comparison = (
    comparison
    .sort_values(
        "macro_f1",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    comparison.round(4)
)

,model,accuracy,balanced_accuracy,macro_f1,weighted_f1,delta_accuracy,delta_balanced_accuracy,delta_macro_f1,delta_weighted_f1
0,role_aware_gate,0.5192,0.4905,0.4949,0.5211,0.0035,0.0091,0.0046,-0.0004
1,context_aware_gate,0.5261,0.4867,0.4946,0.5284,0.0105,0.0053,0.0043,0.0069
2,probability_gate,0.5157,0.4876,0.4931,0.5172,-0.0000,0.0062,0.0029,-0.0043
3,semantic,0.5157,0.4814,0.4903,0.5215,0.0000,0.0000,0.0000,0.0000
4,trajectory_state_soft_gate,0.5192,0.4838,0.4888,0.5232,0.0035,0.0024,-0.0015,0.0017
5,weighted_selective_rescue,0.4983,0.4808,0.4778,0.5075,-0.0174,-0.0006,-0.0125,-0.0140


In [133]:
rescue_summary = pd.DataFrame([
    {
        "router": "unweighted_selective_rescue",
        "threshold": 0.73,
        "overrides": 34,
        "good_overrides": 14,
        "bad_overrides": 12,
        "net_rescues": 2,
        "override_precision": 14 / 34,
    },
    {
        "router": "weighted_selective_rescue",
        "threshold": WEIGHTED_RESCUE_THRESHOLD,
        "overrides": int(weighted_override.sum()),
        "good_overrides": int(good_weighted_override.sum()),
        "bad_overrides": int(bad_weighted_override.sum()),
        "net_rescues": int(
            good_weighted_override.sum()
            - bad_weighted_override.sum()
        ),
        "override_precision": (
            good_weighted_override.sum()
            / weighted_override.sum()
            if weighted_override.sum() > 0
            else 0
        ),
    },
])

display(rescue_summary.round(4))

,router,threshold,overrides,good_overrides,bad_overrides,net_rescues,override_precision
0,unweighted_selective_rescue,0.73,34,14,12,2,0.4118
1,weighted_selective_rescue,0.34,18,4,9,-5,0.2222


# Conclusion — Mixture-of-Experts Routing and Selective Fusion

## Research question

The goal of this experiment was to determine whether the semantic and
structured trajectory representations behave as complementary experts and,
if so, whether a learned routing mechanism can determine when the structured
expert should be trusted over the semantic expert.

The experiments were performed using the canonical group-safe train/test split:

- Train: 1,489 examples
- Test: 287 examples
- Train trajectory groups: 335
- Test trajectory groups: 84
- Group overlap: 0

The primary evaluation metric was Macro F1 because the five failure families
are strongly imbalanced, particularly `reasoning_value_error`.

---

## 1. Individual experts

The semantic expert remained substantially stronger than the structured expert.

| Expert | Accuracy | Balanced Accuracy | Macro F1 | Weighted F1 |
|---|---:|---:|---:|---:|
| Semantic | 0.5157 | 0.4814 | 0.4903 | 0.5215 |
| Structured | 0.4390 | 0.4322 | 0.4347 | 0.4507 |

This establishes an important asymmetry between the experts.

The structured representation is not competitive enough to replace the
semantic representation globally. Any useful MoE mechanism therefore needs
to use the structured expert selectively rather than treating both experts
as equally reliable.

---

## 2. Evidence of expert complementarity

Despite its lower standalone performance, the structured expert was correct
on examples where the semantic expert was wrong.

On the 287-example test set:

| Case | Count | Percentage |
|---|---:|---:|
| Both correct | 97 | 33.8% |
| Semantic only correct | 51 | 17.8% |
| Structured only correct | 29 | 10.1% |
| Both wrong | 110 | 38.3% |

The two experts disagreed on approximately 40.4% of test examples.

Most importantly, there were 29 examples where the structured model was
correct and the semantic model was wrong.

Therefore, the structured representation contains complementary predictive
information even though it is weaker in aggregate.

---

## 3. Oracle experiment

To separate expert complementarity from routing quality, an oracle was
constructed that chooses a correct expert whenever either expert is correct.

Results:

- Semantic accuracy: 0.5157
- Structured accuracy: 0.4390
- Oracle branch accuracy: 0.6167

The oracle therefore provides an upper-bound improvement of approximately:

    0.6167 - 0.5157 = +0.1010 accuracy

relative to the semantic expert.

This is one of the most important findings of the MoE investigation.

It shows that the lack of improvement from fusion is not simply because the
structured expert contains no useful information.

Instead, there is a substantial gap between:

1. expert complementarity, and
2. our ability to identify when each expert should be trusted.

In other words:

> Complementarity exists, but routability is weak.

The central MoE problem is therefore not the absence of a useful second
expert. It is the inability of the available routing features to reliably
identify the structured-only rescue cases.

---

## 4. Late fusion / stacking

A group-safe OOF stacking experiment combined semantic and structured
probabilities.

The stacked model achieved:

- Accuracy: 0.5226
- Balanced Accuracy: 0.4710
- Macro F1: 0.4955
- Weighted F1: 0.5249

Compared with the semantic baseline:

- Accuracy: +0.0070
- Balanced Accuracy: -0.0104
- Macro F1: +0.0052
- Weighted F1: +0.0034

The Macro F1 improvement was small.

Bootstrap analysis produced:

- Mean Macro F1 delta: +0.0022
- 95% CI: [-0.0297, +0.0338]
- P(stack > semantic): 0.5534

Therefore, there was no strong evidence that the observed stacking gain was
stable.

The stack did demonstrate that semantic and structured probabilities can be
combined, but it did not reliably close the gap suggested by the oracle.

---

## 5. Learned routing experiments

Several routing strategies were evaluated.

| Model | Accuracy | Balanced Accuracy | Macro F1 | Weighted F1 |
|---|---:|---:|---:|---:|
| Semantic baseline | 0.5157 | 0.4814 | 0.4903 | 0.5215 |
| Probability gate | 0.5157 | 0.4876 | 0.4931 | 0.5172 |
| Context-aware gate | **0.5261** | 0.4867 | 0.4946 | **0.5284** |
| Role-aware gate | 0.5192 | **0.4905** | **0.4949** | 0.5211 |
| Class-conditional gate | 0.5157 | 0.4824 | 0.4890 | 0.5176 |
| Trajectory-state soft gate | 0.5192 | 0.4838 | 0.4888 | 0.5232 |
| Trajectory-state hard router | 0.5017 | 0.4922 | 0.4842 | 0.5130 |

The strongest Macro F1 among the routing experiments was the role-aware gate:

    0.4903 → 0.4949

or approximately:

    Δ Macro F1 = +0.0046

The context-aware gate achieved the best overall accuracy:

    0.5157 → 0.5261

and the best Weighted F1:

    0.5215 → 0.5284

These improvements are positive but small.

No routing strategy approached the oracle performance.

---

## 6. Router accuracy

The direct routing experiments also revealed an important limitation.

On the subset where expert selection matters:

| Router | Routing Accuracy |
|---|---:|
| Always choose semantic | **0.6375** |
| Role-aware router | 0.5875 |
| Trajectory-state router | 0.5875 |
| Probability router | 0.5750 |
| Context-aware router | 0.5750 |

A learned router therefore did not outperform the simple policy of always
trusting the stronger semantic expert.

This explains why increasingly sophisticated gating mechanisms produced only
small downstream improvements.

The router itself is the bottleneck.

---

## 7. Selective rescue instead of full routing

Because globally choosing between experts was difficult, the problem was
reformulated as selective intervention:

> Keep the semantic prediction by default and use the structured prediction
> only when the router predicts that it can rescue a semantic error.

This is a more conservative MoE formulation because the semantic model remains
the default expert.

### Unweighted rescue router

The threshold was selected using group-safe OOF training predictions and then
frozen before test evaluation.

OOF selected:

- Threshold: 0.73
- Overrides: 201
- Good overrides: 89
- Bad overrides: 67
- Net rescues: +22
- Macro F1: 0.5220

On the untouched test set:

- Overrides: 34
- Good overrides: 14
- Bad overrides: 12
- Net rescues: +2
- Override precision: 0.4118

Final test performance:

- Accuracy: 0.5226
- Balanced Accuracy: 0.4829
- Macro F1: 0.4870
- Weighted F1: 0.5283

Although the router produced two net corrections and improved accuracy, it
reduced Macro F1 relative to the semantic baseline.

This suggests that not all rescues have equal value under the class-balanced
evaluation objective.

---

## 8. Class-weighted selective rescue

The rescue objective was therefore modified to account for class imbalance.

Training class weights were:

| Failure family | Weight |
|---|---:|
| workflow_error | 0.451 |
| constraint_error | 0.939 |
| tool_use_error | 1.257 |
| grounding_state_error | 1.220 |
| reasoning_value_error | 9.606 |

The large weight assigned to `reasoning_value_error` reflects its very small
training support.

### OOF result

The weighted router appeared promising during group-safe OOF evaluation.

The selected threshold was 0.34:

- Overrides: 115
- Good overrides: 58
- Bad overrides: 30
- Net rescues: +28
- Accuracy: 0.5601
- Balanced Accuracy: 0.5020
- Macro F1: 0.5270
- Weighted F1: 0.5560

Compared with the unweighted OOF router, weighting produced fewer interventions
while obtaining more net rescues.

This initially suggested that class-aware selective routing might be a better
solution.

### Held-out test result

However, this behavior did not generalize.

Using the frozen threshold of 0.34:

- Overrides: 18
- Good overrides: 4
- Bad overrides: 9
- Neutral overrides: 5
- Net rescues: -5
- Override precision: 0.2222

Test performance became:

| Metric | Result |
|---|---:|
| Accuracy | 0.4983 |
| Balanced Accuracy | 0.4808 |
| Macro F1 | 0.4778 |
| Weighted F1 | 0.5075 |

Relative to the semantic baseline:

- Accuracy: -0.0174
- Balanced Accuracy: -0.0006
- Macro F1: -0.0125
- Weighted F1: -0.0140

The class-weighted router therefore overestimated the generalizability of its
rescue signal.

This is especially visible in the difference between OOF and test routing:

    OOF:  +28 net rescues
    Test:  -5 net rescues

The weighted rescue experiment therefore does not support further threshold or
class-weight tuning on the current test set.

---

## 9. Calibration evidence

The semantic expert was also slightly better calibrated than the structured
expert.

Log loss:

- Semantic: 1.1404
- Structured: 1.1860

Multiclass Brier score:

- Semantic: 0.12225
- Structured: 0.12562

Thus, there is no evidence that the weaker structured expert can be preferred
simply because its probabilities are more trustworthy.

The semantic branch is both stronger in classification performance and
slightly stronger in probabilistic calibration.

---

# Overall conclusion

The MoE experiments reveal three distinct findings.

### Finding 1 — The experts are complementary

The structured expert is substantially weaker globally, but it correctly
classifies examples that the semantic expert misses.

The oracle accuracy of approximately 0.617 compared with semantic accuracy of
approximately 0.516 demonstrates that useful complementary signal exists.

Therefore, structured trajectory information should not be discarded.

### Finding 2 — Complementarity does not imply routability

Although the experts are complementary, the learned routers could not reliably
predict when the structured expert should replace the semantic expert.

The best routing improvements were small:

    Semantic Macro F1:       0.4903
    Best routing Macro F1:   0.4949

Furthermore, learned routing accuracy remained below an always-semantic
selection policy.

The main bottleneck is therefore expert selection rather than the existence of
expert diversity.

### Finding 3 — More aggressive weighting does not solve the problem

Class weighting produced a promising OOF rescue model:

    OOF Macro F1 = 0.5270
    Net rescues  = +28

but failed on the held-out test set:

    Test Macro F1 = 0.4778
    Net rescues   = -5

This demonstrates that optimizing routing weights and thresholds can produce
apparently strong training/OOF behavior without producing reliable
generalization to unseen trajectory groups.

---

# Research decision

Further handcrafted MoE routing is placed on hold.

The experiments have already considered:

- probability-based routing,
- context-aware routing,
- role-aware routing,
- class-conditional routing,
- trajectory-state routing,
- soft gating,
- hard expert selection,
- selective rescue,
- class-weighted selective rescue,
- and group-safe OOF stacking.

Additional threshold tuning or router weighting on the same test split would
provide decreasing scientific value and would risk indirectly adapting the
research process to the held-out test set.

The evidence instead points toward a representation problem.

Current structured features summarize history through aggregate variables such
as counts, previous-tool indicators, role information, repetition indicators,
and error signals.

These features describe properties of the trajectory but do not preserve the
full temporal ordering of events.

Consequently, trajectories with different dynamics can collapse into similar
structured representations.

---

# Next research direction — Learned trajectory representations

The next experiment will therefore stop asking:

> Which existing expert should the model route this example to?

and instead ask:

> Can the model learn a useful representation directly from the ordered
> trajectory history?

The next stage will model the trajectory as a sequence rather than as a set of
aggregate statistics.

A first experiment should compare:

1. Current-message semantic representation
2. Learned historical-sequence representation
3. Current-message semantic + historical-sequence representation

A small sequential encoder such as a GRU provides an appropriate first
baseline before introducing a trajectory Transformer.

The key hypothesis becomes:

> Failure classification depends not only on the semantic content of the
> current message, but also on the ordered sequence of states and actions that
> produced that message.

If a learned trajectory representation improves over the current semantic
baseline under the same group-safe evaluation protocol, this would provide
evidence that trajectory dynamics contain useful information that the
hand-engineered structured features and MoE routers were unable to recover.

If it does not improve performance, this would provide evidence that the
remaining oracle gap cannot be easily recovered from trajectory history under
the current dataset and task formulation.